# 113 — CPTAC-3 External Cohort Construction and Eligibility

## Scientific objective

Construct an auditable, outcome-blind external RNA-seq source cohort from
the GDC CPTAC-3 project for subsequent lineage-aware evaluation of
previously defined transcriptomic programs.

The analysis:

1. Identifies uniquely represented primary-tumor RNA-seq cases while
   preserving case, sample, aliquot, and file identities.
2. Establishes source provenance and documents the qualified assumption
   of cohort-level externality relative to the frozen TCGA discovery population.
3. Harmonizes source tumor contexts with supported TCGA cancer lineages,
   without inferring unresolved histological subtypes.
4. Evaluates technical suitability using audited source information and
   verifies compatibility with the frozen transcriptomic feature universe.
5. Constructs a proposed external source cohort with explicit eligibility
   criteria, non-inclusion reasons, and lineage-specific denominators.

All eligibility decisions are independent of transcriptomic program scores
and downstream replication outcomes.

## Data sources

The analysis uses:

- **Notebook 111:** audited GDC CPTAC-3 acquisition records, source
  identities, sample ancestry, and file-integrity evidence.
- **Notebook 112:** source-level RNA-seq technical diagnostics and available
  sequencing-quality metadata.
- **Frozen TCGA reference:** the registered primary-tumor sample mapping
  (`phase2.203.final_sample_mapping`).
- **Frozen Phase 4 reference:** the immutable transcriptomic feature
  universe associated with the previously defined consensus programs.

Source identities and reference artifacts are verified using their
registered provenance and fingerprints. Previously audited acquisition
and technical assessments are reused without unnecessary repetition.

## Eligibility and interpretation

Primary candidates require a uniquely resolved primary-tumor RNA-seq
file, sample, and aliquot. Proposed cohort membership additionally
requires supported cancer-lineage correspondence, compatible raw-count
inputs, and the documented cohort-level provenance assumption.

Cases with unresolved eligibility conditions remain accounted for,
with explicit classification and non-inclusion reasons.

GDC CPTAC-3 provenance supports a qualified external study-design
assumption; it does not establish individual donor independence from TCGA.
Likewise, source-file integrity and raw-count compatibility do not
constitute independent read-level sequencing QC certification.

The resulting cohort is a proposed, technically characterized source
population, not a frozen or authorized inferential validation cohort.

## Reproducible outputs

The notebook produces four derived artifacts under
`data/interim/metadata/`:

- `113_proposed_external_source_cohort.csv` — proposed cohort membership,
  source identities, and accepted cancer lineages.
- `113_primary_candidate_case_eligibility.csv` — complete candidate ledger,
  eligibility decisions, provenance, and non-inclusion reasons.
- `113_raw_count_compatibility.csv` — file-level raw-count and frozen-feature
  compatibility assessments.
- `113_source_provenance_manifest.json` — source provenance, input and output
  fingerprints, cohort status, and analytical limitations.

Descriptive summaries and intermediate diagnostics remain within the
notebook unless required as reproducible downstream inputs.

## Scientific boundary

This notebook establishes source-cohort eligibility and transcriptomic
input compatibility. It does not evaluate program replication, predict
drug resistance, infer causal mechanisms, or establish therapeutic effects.

External structural replication requires a separately specified,
lineage-aware analysis with appropriate normalization, statistical
inference, negative controls, and confounding assessments.

In [ ]:
# =============================================================================
# Imports, repository paths, and validation utilities
# =============================================================================

import json
import hashlib
import time
from datetime import datetime, timezone

import requests

import numpy as np
import pandas as pd
from IPython.display import display

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths

# Repository locations
ROOT = Paths.root
META = Paths.metadata
REGISTRY = ROOT / "config" / "artifact_registry.json"

# Audited CPTAC-3 source inputs
SOURCE_FILE_INDEX = META / "111_cptac3_file_index.csv"
PRIMARY_TUMOR_CANDIDATES = (
    META / "111_cptac3_provisional_primary_tumor_files.csv"
)

def require_file(path):
    """Require an existing input without modifying the source."""
    if not path.is_file():
        raise FileNotFoundError(f"Required input file not found: {path}")

    return path


def load_csv(path):
    """Load a source table while preserving identifiers as strings."""
    return pd.read_csv(
        require_file(path),
        dtype="string",
    )


def require_unique(df, columns, label):
    """Reject missing or duplicated identifiers."""
    if df[columns].isna().any().any():
        raise ValueError(f"{label}: missing identifiers in {columns}")

    if df.duplicated(subset=columns).any():
        raise ValueError(f"{label}: duplicated identifiers in {columns}")


def normalize_identifier(values):
    """Normalize identifier formatting for explicit comparisons."""
    return values.astype("string").str.strip().str.casefold()


def sha256(path):
    """Calculate the SHA-256 fingerprint of an existing file."""
    return calculate_sha256(require_file(path))

In [2]:
# =============================================================================
# Load audited CPTAC-3 inputs and verify the frozen TCGA reference
# =============================================================================

# Load the authoritative artifact registry
with require_file(REGISTRY).open(encoding="utf-8") as handle:
    registry = json.load(handle)

tcga_reference = registry["artifacts"]["phase2.203.final_sample_mapping"]

if tcga_reference["status"] != "frozen":
    raise ValueError("TCGA reference is not frozen")

if tcga_reference["shape"][0] != 9965:
    raise ValueError("Unexpected TCGA reference case count")

# Verify the registered TCGA reference fingerprint
TCGA_REFERENCE_PATH = require_file(ROOT / tcga_reference["path"])

if sha256(TCGA_REFERENCE_PATH) != tcga_reference["sha256"]:
    raise ValueError("Frozen TCGA reference SHA-256 mismatch")

# Load audited source and reference tables
source = load_csv(SOURCE_FILE_INDEX)
primary = load_csv(PRIMARY_TUMOR_CANDIDATES)
tcga = load_csv(TCGA_REFERENCE_PATH)

# Verify the loaded reference dimensions
if len(tcga) != tcga_reference["shape"][0]:
    raise ValueError("Loaded TCGA reference row count mismatch")

print("Audited input datasets:")
print(f"  CPTAC-3 file index:        {len(source):,} rows")
print(f"  Primary-tumor candidates:  {len(primary):,} files")
print(f"  Frozen TCGA reference:     {len(tcga):,} cases")
print("  TCGA reference SHA-256:    VERIFIED")

Audited input datasets:
  CPTAC-3 file index:        5,578 rows
  Primary-tumor candidates:  3,181 files
  Frozen TCGA reference:     9,965 cases
  TCGA reference SHA-256:    VERIFIED


## Primary-Tumor RNA-seq Candidate Selection

Identify cases with a single, uniquely mapped primary-tumor RNA-seq file, sample, and aliquot using the audited CPTAC-3 source records.

Cases with multiple primary samples or aliquots are excluded from the primary candidate universe. Additional files with ambiguous sample ancestry are flagged without automatically excluding otherwise eligible cases.

The resulting candidate cohort is evaluated independently of downstream transcriptomic program outcomes.

In [3]:
# =============================================================================
# Identify uniquely represented primary-tumor RNA-seq candidates
# =============================================================================

required_columns = {
    "modality",
    "file_id",
    "case_uuid",
    "case_submitter_id",
    "sample_uuid",
    "aliquot_uuid",
    "mapping_status",
    "sample_type",
    "primary_site",
    "disease_type",
    "project_id",
    "provider_md5",
}

for label, frame in [("Source inventory", source), ("Primary inventory", primary)]:
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{label}: missing columns {sorted(missing)}")

    require_unique(frame, ["modality", "file_id"], label)

# Verify the audited source universe
rna = source.loc[source["modality"].eq("rnaseq")].copy()

if len(rna) != 2968:
    raise ValueError("Unexpected RNA-seq source file count")

if source["modality"].eq("methylation").sum() != 2610:
    raise ValueError("Unexpected methylation source file count")

# Retain uniquely resolved primary-tumor RNA-seq files
candidate = primary.loc[primary["modality"].eq("rnaseq")].copy()

if len(candidate) != 1731 or candidate["case_uuid"].nunique() != 1370:
    raise ValueError("Primary-tumor source universe differs from the audited reference")

if not candidate["file_id"].isin(rna["file_id"]).all():
    raise ValueError("Primary RNA-seq files are missing from the source inventory")

if not candidate["mapping_status"].eq("unique_aliquot_sample").all():
    raise ValueError("Unresolved sample ancestry in primary candidates")

if not candidate["sample_type"].eq("Primary Tumor").all():
    raise ValueError("Unexpected sample type in primary candidates")

identity_columns = ["file_id", "case_uuid", "sample_uuid", "aliquot_uuid"]
require_unique(candidate, ["file_id"], "Primary RNA-seq candidates")

if candidate[identity_columns].isna().any().any():
    raise ValueError("Missing primary-tumor source identifiers")

# Classify primary-tumor representation at case level
case_count = candidate.groupby("case_uuid").agg(
    n_files=("file_id", "nunique"),
    n_samples=("sample_uuid", "nunique"),
    n_aliquots=("aliquot_uuid", "nunique"),
)

simple_mask = (
    case_count["n_files"].eq(1)
    & case_count["n_samples"].eq(1)
    & case_count["n_aliquots"].eq(1)
)

simple_cases = case_count.index[simple_mask]
mult = case_count.loc[~simple_mask]

if len(case_count) != 1370 or len(simple_cases) != 1082 or len(mult) != 288:
    raise ValueError("Primary-tumor case classification mismatch")

if (
    mult["n_samples"].gt(1).sum() != 276
    or (mult["n_samples"].eq(1) & mult["n_aliquots"].gt(1)).sum() != 12
    or mult["n_files"].sum() != 649
):
    raise ValueError("Primary-tumor multiplicity classification mismatch")

# Build the primary candidate ledger
ledger = (
    candidate.loc[candidate["case_uuid"].isin(simple_cases)]
    .sort_values("case_uuid")
    .reset_index(drop=True)
)

require_unique(ledger, ["case_uuid"], "Primary candidate ledger")

if len(ledger) != 1082:
    raise ValueError("Unexpected primary candidate count")

# Preserve additional ambiguous RNA-seq ancestry as a sensitivity flag
ambiguous_rna = rna.loc[
    rna["mapping_status"].eq("ambiguous_sample_ancestry")
]

if len(ambiguous_rna) != 314:
    raise ValueError("Unexpected ambiguous RNA-seq file count")

flagged_cases = set(ambiguous_rna["case_uuid"].dropna()) & set(
    ledger["case_uuid"]
)

if len(flagged_cases) != 37:
    raise ValueError("Unexpected number of cases with additional ambiguous RNA files")

ledger["other_ambiguous_rna_flag"] = ledger["case_uuid"].isin(flagged_cases)
ledger["primary_candidate_status"] = "SIMPLE_PRIMARY_CANDIDATE"

# Summarize audited source selection
print("Primary-tumor RNA-seq candidate selection:")
print(f"  Primary source cases:          {len(case_count):,}")
print(f"  Uniquely represented cases:    {len(ledger):,}")
print(f"  Cases with multiple samples:   {mult['n_samples'].gt(1).sum():,}")
print(
    "  Cases with multiple aliquots:  "
    f"{(mult['n_samples'].eq(1) & mult['n_aliquots'].gt(1)).sum():,}"
)
print(f"  Additional ambiguity flags:   {len(flagged_cases):,}")
print("  Selection criteria:           OUTCOME-BLIND")

Primary-tumor RNA-seq candidate selection:
  Primary source cases:          1,370
  Uniquely represented cases:    1,082
  Cases with multiple samples:   276
  Cases with multiple aliquots:  12
  Additional ambiguity flags:   37
  Selection criteria:           OUTCOME-BLIND


## Source Multiplicity and Specimen Metadata Assessment

Characterize primary-tumor sample and aliquot multiplicity, and assess the availability of specimen-level metadata in the audited CPTAC-3 records.

Cases with multiple primary-tumor candidates remain outside the primary eligibility universe. Metadata availability alone does not justify selecting a representative sample or establishing sequencing QC eligibility.

In [4]:
# =============================================================================
# Classify primary-tumor RNA-seq sample and aliquot multiplicity
# =============================================================================

rna_cases = case_count.reset_index()

rna_cases["needs_selection_policy"] = rna_cases[
    ["n_files", "n_samples", "n_aliquots"]
].gt(1).any(axis=1)

invalid_counts = (
    rna_cases[["n_files", "n_samples", "n_aliquots"]].le(0).any(axis=1)
    | rna_cases["n_aliquots"].gt(rna_cases["n_files"])
    | rna_cases["n_samples"].gt(rna_cases["n_aliquots"])
)

rna_cases["rna_multiplicity_class"] = np.select(
    [
        invalid_counts,
        rna_cases["n_samples"].gt(1),
        rna_cases["n_aliquots"].gt(1),
        rna_cases["n_files"].gt(1),
    ],
    [
        "INCONSISTENT_BIOSPECIMEN_COUNTS",
        "MULTIPLE_SAMPLES_IN_CASE",
        "ONE_SAMPLE_MULTIPLE_ALIQUOTS",
        "ONE_ALIQUOT_MULTIPLE_FILES",
    ],
    default="SINGLE_FILE_SAMPLE_ALIQUOT",
)

display(
    rna_cases["rna_multiplicity_class"]
    .value_counts()
    .rename_axis("multiplicity_class")
    .reset_index(name="n_cases")
)

,multiplicity_class,n_cases
0,SINGLE_FILE_SAMPLE_ALIQUOT,1082
1,MULTIPLE_SAMPLES_IN_CASE,276
2,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12


In [5]:
# =============================================================================
# Characterize specimen identifiers and source metadata
# =============================================================================

def joined_unique(values):
    return "|".join(sorted(set(values.dropna())))


specimen_metadata = candidate.groupby("case_uuid").agg(
    aliquot_uuid_values=("aliquot_uuid", joined_unique),
    sample_uuid_values=("sample_uuid", joined_unique),
    primary_site_values=("primary_site", joined_unique),
    disease_type_values=("disease_type", joined_unique),
)

rna_cases = rna_cases.merge(
    specimen_metadata,
    on="case_uuid",
    how="left",
    validate="one_to_one",
)

print("Primary-tumor cases:", len(rna_cases))
print("Cases requiring multiplicity resolution:", int(rna_cases["needs_selection_policy"].sum()))

Primary-tumor cases: 1370
Cases requiring multiplicity resolution: 288


In [6]:
# =============================================================================
# Identify conflicting biospecimen parent relationships
# =============================================================================

aliquot_links = candidate.groupby("aliquot_uuid").agg(
    n_parent_samples=("sample_uuid", "nunique"),
    n_parent_cases=("case_uuid", "nunique"),
)

sample_links = candidate.groupby("sample_uuid").agg(
    n_parent_cases=("case_uuid", "nunique"),
)

conflicting_aliquots = set(
    aliquot_links.index[
        aliquot_links["n_parent_samples"].gt(1)
        | aliquot_links["n_parent_cases"].gt(1)
    ]
)

conflicting_samples = set(
    sample_links.index[sample_links["n_parent_cases"].gt(1)]
)

print("Aliquots with conflicting parents:", len(conflicting_aliquots))
print("Samples linked to multiple cases:", len(conflicting_samples))

Aliquots with conflicting parents: 0
Samples linked to multiple cases: 0


In [7]:
# =============================================================================
# Flag cases with conflicting biospecimen ancestry
# =============================================================================

case_conflicts = candidate.assign(
    ancestry_conflict=(
        candidate["aliquot_uuid"].isin(conflicting_aliquots)
        | candidate["sample_uuid"].isin(conflicting_samples)
    )
).groupby("case_uuid")["ancestry_conflict"].any()

rna_cases["cross_file_ancestry_conflict"] = (
    rna_cases["case_uuid"].map(case_conflicts).fillna(False).astype(bool)
)

rna_cases.loc[
    rna_cases["cross_file_ancestry_conflict"],
    "rna_multiplicity_class",
] = "CROSS_FILE_BIOSPECIMEN_CONFLICT"

print(
    "Cases with conflicting ancestry:",
    int(rna_cases["cross_file_ancestry_conflict"].sum()),
)

Cases with conflicting ancestry: 0


In [8]:
# =============================================================================
# Prepare source metadata for cases with primary RNA-seq multiplicity
# =============================================================================

rna_review_cases = rna_cases.loc[
    rna_cases["needs_selection_policy"],
    [
        "case_uuid",
        "rna_multiplicity_class",
        "n_files",
        "n_samples",
        "n_aliquots",
        "cross_file_ancestry_conflict",
    ],
].copy()

source_evidence_columns = [
    "case_uuid",
    "file_id",
    "filename",
    "sample_uuid",
    "sample_submitter_id",
    "aliquot_uuid",
    "aliquot_submitter_id",
    "sample_type",
    "tissue_type",
    "tumor_descriptor",
    "primary_site",
    "disease_type",
    "project_id",
    "workflow_type",
    "platform",
    "size_bytes",
    "provider_md5",
]

missing_columns = set(source_evidence_columns) - set(candidate.columns)
if missing_columns:
    raise ValueError(f"Missing source metadata: {sorted(missing_columns)}")

rna_adjudication_files = candidate.loc[
    candidate["case_uuid"].isin(rna_review_cases["case_uuid"]),
    source_evidence_columns,
].copy()

print(f"Cases requiring multiplicity review: {len(rna_review_cases):,}")
print(f"Primary RNA-seq files under review: {len(rna_adjudication_files):,}")

Cases requiring multiplicity review: 288
Primary RNA-seq files under review: 649


In [9]:
# =============================================================================
# Assess specimen identifier and descriptor coverage
# =============================================================================

evidence = rna_adjudication_files.groupby("case_uuid").agg(
    observed_files=("file_id", "nunique"),
    observed_samples=("sample_uuid", "nunique"),
    observed_aliquots=("aliquot_uuid", "nunique"),
    sample_submitter_ids=("sample_submitter_id", "nunique"),
    sample_submitter_id_file_coverage=("sample_submitter_id", "count"),
    aliquot_submitter_ids=("aliquot_submitter_id", "nunique"),
    aliquot_submitter_id_file_coverage=("aliquot_submitter_id", "count"),
    tissue_type_values=("tissue_type", "nunique"),
    tissue_type_file_coverage=("tissue_type", "count"),
    tumor_descriptor_values=("tumor_descriptor", "nunique"),
    tumor_descriptor_file_coverage=("tumor_descriptor", "count"),
)

In [10]:
# =============================================================================
# Characterize source and technical metadata
# =============================================================================

source_metadata = rna_adjudication_files.groupby("case_uuid").agg(
    source_site_values=("primary_site", "nunique"),
    source_disease_values=("disease_type", "nunique"),
    project_values=("project_id", "nunique"),
    workflow_values=("workflow_type", "nunique"),
    platform_values=("platform", "nunique"),
    distinct_provider_md5=("provider_md5", "nunique"),
)

evidence = evidence.join(source_metadata).reset_index()

In [11]:
# =============================================================================
# Summarize metadata coverage by multiplicity class
# =============================================================================

rna_adjudication_cases = rna_review_cases.merge(
    evidence,
    on="case_uuid",
    how="left",
    validate="one_to_one",
)

for expected, observed in [
    ("n_files", "observed_files"),
    ("n_samples", "observed_samples"),
    ("n_aliquots", "observed_aliquots"),
]:
    if not rna_adjudication_cases[expected].eq(
        rna_adjudication_cases[observed]
    ).all():
        raise ValueError(f"Specimen count mismatch: {observed}")

coverage_columns = [
    "observed_files",
    "tissue_type_file_coverage",
    "tumor_descriptor_file_coverage",
    "sample_submitter_id_file_coverage",
    "aliquot_submitter_id_file_coverage",
]

display(
    rna_adjudication_cases.groupby("rna_multiplicity_class")[
        coverage_columns
    ].sum()
)

,observed_files,tissue_type_file_coverage,tumor_descriptor_file_coverage,sample_submitter_id_file_coverage,aliquot_submitter_id_file_coverage
rna_multiplicity_class,,,,,
MULTIPLE_SAMPLES_IN_CASE,619,619,619,619,619
ONE_SAMPLE_MULTIPLE_ALIQUOTS,30,30,30,30,30


In [12]:
# =============================================================================
# Assess specimen metadata consistency and completeness
# =============================================================================

descriptor_columns = [
    "tissue_type_values",
    "tumor_descriptor_values",
    "source_site_values",
    "source_disease_values",
    "project_values",
    "workflow_values",
    "platform_values",
]

coverage_columns = [
    "tissue_type_file_coverage",
    "tumor_descriptor_file_coverage",
    "sample_submitter_id_file_coverage",
    "aliquot_submitter_id_file_coverage",
]

rna_adjudication_cases["source_descriptor_discordance"] = (
    rna_adjudication_cases[descriptor_columns].gt(1).any(axis=1)
)

rna_adjudication_cases["source_descriptor_incomplete"] = (
    rna_adjudication_cases[coverage_columns]
    .lt(rna_adjudication_cases["n_files"], axis=0)
    .any(axis=1)
)

rna_adjudication_cases["same_md5_across_distinct_files"] = (
    rna_adjudication_cases["distinct_provider_md5"]
    .lt(rna_adjudication_cases["n_files"])
)

In [13]:
# =============================================================================
# Classify evidence limitations without selecting source files
# =============================================================================

rna_adjudication_cases["adjudication_evidence_class"] = np.select(
    [
        rna_adjudication_cases["cross_file_ancestry_conflict"],
        rna_adjudication_cases["source_descriptor_discordance"],
        rna_adjudication_cases["source_descriptor_incomplete"],
        rna_adjudication_cases["rna_multiplicity_class"].eq(
            "MULTIPLE_SAMPLES_IN_CASE"
        ),
    ],
    [
        "CROSS_FILE_CONFLICT_UNRESOLVED",
        "SOURCE_DESCRIPTORS_DISCORDANT_REVIEW",
        "SOURCE_DESCRIPTORS_INCOMPLETE_REVIEW",
        "MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS",
    ],
    default="MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS",
)

rna_adjudication_cases["selection_evidence_status"] = (
    "INSUFFICIENT_TO_SELECT_WITHOUT_PROSPECTIVE_QC_POLICY"
)

rna_adjudication_cases["selected_file_id"] = pd.Series(
    pd.NA,
    index=rna_adjudication_cases.index,
    dtype="string",
)

display(
    rna_adjudication_cases["adjudication_evidence_class"]
    .value_counts(dropna=False)
    .rename_axis("evidence_class")
    .reset_index(name="n_cases")
)

,evidence_class,n_cases
0,MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS,276
1,MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS,12


In [14]:
# =============================================================================
# Associate source tumor context with multiplicity assessments
# =============================================================================

rna_case_context = rna_cases.loc[
    rna_cases["needs_selection_policy"],
    ["case_uuid", "primary_site_values", "disease_type_values"],
]

rna_adjudication_cases = rna_adjudication_cases.merge(
    rna_case_context,
    on="case_uuid",
    validate="one_to_one",
)

In [15]:
# =============================================================================
# Summarize multiplicity evidence by source tumor context
# =============================================================================

context_summary = (
    rna_adjudication_cases.groupby(
        ["primary_site_values", "rna_multiplicity_class"],
        dropna=False,
    )
    .agg(
        n_cases=("case_uuid", "nunique"),
        descriptor_discordance=("source_descriptor_discordance", "sum"),
        incomplete_descriptors=("source_descriptor_incomplete", "sum"),
        same_md5_file_groups=("same_md5_across_distinct_files", "sum"),
    )
    .reset_index()
    .sort_values(
        ["n_cases", "primary_site_values"],
        ascending=[False, True],
    )
)

display(context_summary.head(25))

print(
    "Cases with descriptor discordance:",
    int(rna_adjudication_cases["source_descriptor_discordance"].sum()),
)
print(
    "Cases with incomplete descriptors:",
    int(rna_adjudication_cases["source_descriptor_incomplete"].sum()),
)
print("Technical replicate identity and relative RNA-seq QC: NOT ASSESSED")

,primary_site_values,rna_multiplicity_class,n_cases,descriptor_discordance,incomplete_descriptors,same_md5_file_groups
13,"Uterus, NOS",MULTIPLE_SAMPLES_IN_CASE,179,0,0,0
4,Kidney,MULTIPLE_SAMPLES_IN_CASE,40,0,0,0
5,Larynx,MULTIPLE_SAMPLES_IN_CASE,19,0,0,0
11,Other and unspecified parts of tongue,MULTIPLE_SAMPLES_IN_CASE,12,0,0,0
12,Pancreas,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,0,0,0
2,Floor of mouth,MULTIPLE_SAMPLES_IN_CASE,11,0,0,0
3,Gum,MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
7,Oropharynx,MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
9,"Other and ill-defined sites in lip, oral cavit...",MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
6,Lip,MULTIPLE_SAMPLES_IN_CASE,2,0,0,0


Cases with descriptor discordance: 0
Cases with incomplete descriptors: 0
Technical replicate identity and relative RNA-seq QC: NOT ASSESSED


In [16]:
# =============================================================================
# Finalize specimen-level evidence for unresolved multiplicity
# =============================================================================

rna_adjudication_files = rna_adjudication_files.merge(
    rna_adjudication_cases[
        [
            "case_uuid",
            "rna_multiplicity_class",
            "adjudication_evidence_class",
            "selection_evidence_status",
        ]
    ],
    on="case_uuid",
    validate="many_to_one",
)

rna_adjudication_files["candidate_selected"] = False
rna_adjudication_files["selection_performed"] = False

rna_adjudication_cases = rna_adjudication_cases.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)

rna_adjudication_files = rna_adjudication_files.sort_values(
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"],
    kind="stable",
).reset_index(drop=True)

if rna_adjudication_cases["selected_file_id"].notna().any():
    raise ValueError("Unexpected file selection in multiplicity review")

print(f"Cases with unresolved multiplicity: {len(rna_adjudication_cases):,}")
print(f"Primary RNA-seq files involved:     {len(rna_adjudication_files):,}")
print(
    "Representative files selected:     "
    f"{int(rna_adjudication_files['candidate_selected'].sum()):,}"
)

Cases with unresolved multiplicity: 288
Primary RNA-seq files involved:     649
Representative files selected:     0


In [17]:
# =============================================================================
# Load and authenticate audited GDC RNA-seq metadata
# =============================================================================

RNA_GDC_METADATA_PATH = (
    Paths.config / "manifests" / "cptac3" / "gdc_metadata_cptac3_rnaseq.json"
)

AUDIT_SUMMARY_PATH = META / "111_cptac3_acquisition_audit_summary.json"

with require_file(AUDIT_SUMMARY_PATH).open(encoding="utf-8") as handle:
    audit_summary = json.load(handle)

expected_sha = audit_summary[
    "source_fingerprints"
]["rnaseq"]["metadata"]["sha256"]

actual_sha = sha256(RNA_GDC_METADATA_PATH)

if actual_sha != expected_sha:
    raise ValueError("GDC RNA-seq metadata fingerprint mismatch")

with RNA_GDC_METADATA_PATH.open(encoding="utf-8") as handle:
    rna_snapshot = json.load(handle)

if not isinstance(rna_snapshot, dict) or not isinstance(
    rna_snapshot.get("data"), dict
):
    raise ValueError("Invalid GDC RNA-seq metadata structure")

rna_metadata_hits = rna_snapshot["data"].get("hits")

if not isinstance(rna_metadata_hits, list):
    raise ValueError("GDC RNA-seq metadata records are missing")

print(f"GDC RNA-seq metadata records: {len(rna_metadata_hits):,}")
print(f"Snapshot SHA-256 verified:    {actual_sha == expected_sha}")

GDC RNA-seq metadata records: 2,968
Snapshot SHA-256 verified:    True


In [18]:
# =============================================================================
# Match audited GDC metadata to multiplicity-review files
# =============================================================================

review_file_ids = set(rna_adjudication_files["file_id"])

rna_c_hits = {}

for hit in rna_metadata_hits:
    file_uuid = hit.get("file_id") or hit.get("id")

    if file_uuid not in review_file_ids:
        continue

    if file_uuid in rna_c_hits:
        raise ValueError(f"Duplicate GDC metadata file ID: {file_uuid}")

    rna_c_hits[file_uuid] = hit

missing_file_ids = review_file_ids - set(rna_c_hits)

if missing_file_ids:
    raise ValueError(
        f"Missing GDC metadata for {len(missing_file_ids)} reviewed files"
    )

print(f"Files requiring metadata review: {len(review_file_ids):,}")
print(f"Matched GDC metadata records:    {len(rna_c_hits):,}")
print(f"Files without GDC metadata:      {len(missing_file_ids):,}")

Files requiring metadata review: 649
Matched GDC metadata records:    649
Files without GDC metadata:      0


In [19]:
# =============================================================================
# Resolve exact case–sample–aliquot paths in GDC metadata
# =============================================================================

def exact_source_nodes(hit, case_uuid, sample_uuid, aliquot_uuid):
    paths = []

    for case in hit.get("cases") or []:
        if case.get("case_id") != case_uuid:
            continue

        for sample in case.get("samples") or []:
            if sample.get("sample_id") != sample_uuid:
                continue

            for portion in sample.get("portions") or []:
                for analyte in portion.get("analytes") or []:
                    for aliquot in analyte.get("aliquots") or []:
                        if aliquot.get("aliquot_id") == aliquot_uuid:
                            paths.append(
                                {
                                    "sample": sample,
                                    "portion": portion,
                                    "analyte": analyte,
                                    "aliquot": aliquot,
                                }
                            )

    return paths

In [20]:
# =============================================================================
# Extract source metadata nodes for reviewed RNA-seq files
# =============================================================================

rna_c_node_maps = {}
rna_c_path_rows = []

for row in rna_adjudication_files.itertuples(index=False):
    hit = rna_c_hits[row.file_id]

    paths = exact_source_nodes(
        hit, row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )

    if not paths:
        raise ValueError(f"Missing exact GDC hierarchy path: {row.file_id}")

    rna_c_node_maps[row.file_id] = {
        "file": [hit],
        "analysis": (
            [hit["analysis"]]
            if isinstance(hit.get("analysis"), dict)
            else []
        ),
        **{
            scope: [path[scope] for path in paths]
            for scope in ("sample", "portion", "analyte", "aliquot")
        },
    }

    rna_c_path_rows.append(
        {
            "file_id": row.file_id,
            "case_uuid": row.case_uuid,
            "sample_uuid": row.sample_uuid,
            "aliquot_uuid": row.aliquot_uuid,
            "exact_hierarchy_path_rows": len(paths),
        }
    )

In [21]:
# =============================================================================
# Summarize exact GDC hierarchy path coverage
# =============================================================================

rna_c_exact_path_index = pd.DataFrame(rna_c_path_rows)

print(f"Files with exact hierarchy paths: {len(rna_c_exact_path_index):,}")
print(
    "Files with multiple exact hierarchy rows: "
    f"{int(rna_c_exact_path_index['exact_hierarchy_path_rows'].gt(1).sum()):,}"
)
print(
    "Files without exact hierarchy paths: "
    f"{int(rna_c_exact_path_index['exact_hierarchy_path_rows'].eq(0).sum()):,}"
)

Files with exact hierarchy paths: 649
Files with multiple exact hierarchy rows: 0
Files without exact hierarchy paths: 0


In [22]:
# =============================================================================
# Define sample descriptors and collection metadata probes
# =============================================================================

RNA_C_FIELDS = {
    "generic_sample_descriptors": [
        ("sample", "sample_type"),
        ("sample", "tissue_type"),
        ("sample", "tumor_descriptor"),
    ],
    "collection_timing": [
        ("sample", "days_to_collection"),
        ("sample", "days_to_sample_procurement"),
        ("sample", "time_between_clamping_and_freezing"),
        ("sample", "time_between_excision_and_freezing"),
    ],
    "sampling_location_detail": [
        ("sample", "tumor_region"),
        ("sample", "tumor_subsite"),
        ("sample", "sample_collection_site"),
        ("sample", "anatomic_site"),
        ("sample", "specimen_laterality"),
    ],
}

In [23]:
# =============================================================================
# Define processing provenance and identifier probes
# =============================================================================

RNA_C_FIELDS.update(
    {
        "processing_provenance": [
            ("sample", "method_of_sample_procurement"),
            ("sample", "preservation_method"),
            ("sample", "is_ffpe"),
            ("portion", "is_ffpe"),
            ("portion", "portion_number"),
            ("portion", "weight"),
            ("analyte", "analyte_type"),
            ("analyte", "concentration"),
            ("aliquot", "concentration"),
        ],
        "identifiers_not_qc": [
            ("sample", "submitter_id"),
            ("portion", "submitter_id"),
            ("analyte", "submitter_id"),
            ("aliquot", "submitter_id"),
        ],
    }
)

In [24]:
# =============================================================================
# Define file processing and sequencing metadata probes
# =============================================================================

RNA_C_FIELDS.update(
    {
        "file_processing_dates_not_collection": [
            ("file", "created_datetime"),
            ("file", "updated_datetime"),
            ("analysis", "analysis_id"),
            ("analysis", "workflow_type"),
            ("analysis", "workflow_version"),
        ],
        "quantitative_qc_metadata_probes": [
            ("file", "total_reads"),
            ("file", "mapped_reads"),
            ("file", "average_read_length"),
            ("file", "average_base_quality"),
            ("analysis", "total_reads"),
            ("analysis", "mapped_reads"),
            ("analysis", "mapping_rate"),
            ("analysis", "duplication_rate"),
            ("analysis", "percent_aligned_reads"),
        ],
    }
)

In [25]:
# =============================================================================
# Define source metadata availability assessment
# =============================================================================

PLACEHOLDER_STRINGS = {
    "", "unknown", "not reported", "not available",
    "not applicable", "n/a", "na", "null",
    "none", "unspecified", "--",
}


def informative_scalar(value):
    if value is None or isinstance(value, (list, tuple, dict)):
        return False
    if isinstance(value, str):
        return value.strip().casefold() not in PLACEHOLDER_STRINGS
    if isinstance(value, (bool, int, float)):
        return bool(pd.notna(value))
    return False


def source_value_status(nodes, key):
    values = [
        item[key]
        for item in nodes
        if isinstance(item, dict) and key in item
    ]
    useful = {str(v).strip() for v in values if informative_scalar(v)}
    return bool(values), bool(useful), len(useful) > 1

In [26]:
# =============================================================================
# Summarize source metadata availability probes
# =============================================================================

all_probes = {
    (scope, key)
    for group in RNA_C_FIELDS.values()
    for scope, key in group
}

print(f"Metadata domains: {len(RNA_C_FIELDS):,}")
print(f"Distinct metadata probes: {len(all_probes):,}")

Metadata domains: 7
Distinct metadata probes: 39


In [27]:
# =============================================================================
# Identify scalar metadata fields in the GDC source hierarchy
# =============================================================================

observed_scalar_keys = set()

for node_map in rna_c_node_maps.values():
    for scope, nodes in node_map.items():
        for node in nodes:
            if isinstance(node, dict):
                observed_scalar_keys.update(
                    (scope, key)
                    for key, value in node.items()
                    if not isinstance(value, (dict, list, tuple))
                )

all_field_keys = sorted(all_probes | observed_scalar_keys)

print(f"Distinct observed scalar fields: {len(observed_scalar_keys):,}")
print(f"Fields included in the inventory: {len(all_field_keys):,}")

Distinct observed scalar fields: 18
Fields included in the inventory: 50


In [28]:
# =============================================================================
# Quantify metadata field availability across reviewed RNA-seq files
# =============================================================================

rna_c_field_records = []

for scope, key in all_field_keys:
    statuses = [
        source_value_status(nodes[scope], key)
        for nodes in rna_c_node_maps.values()
    ]

    assigned_domains = sorted(
        name
        for name, fields in RNA_C_FIELDS.items()
        if (scope, key) in fields
    )

    rna_c_field_records.append(
        {
            "scope": scope,
            "field": key,
            "planned_domains": "|".join(assigned_domains),
            "observed_scalar_key": (scope, key) in observed_scalar_keys,
            "n_files_with_key": sum(status[0] for status in statuses),
            "n_files_with_informative_scalar": sum(
                status[1] for status in statuses
            ),
            "n_files_with_multiple_informative_values": sum(
                status[2] for status in statuses
            ),
            "n_reviewed_files": len(rna_c_node_maps),
        }
    )

rna_c_field_coverage = pd.DataFrame(rna_c_field_records)

In [29]:
# =============================================================================
# Summarize planned metadata probes and additional observed fields
# =============================================================================

planned_field_coverage = (
    rna_c_field_coverage.loc[
        rna_c_field_coverage["planned_domains"].ne("")
    ]
    .sort_values(["planned_domains", "scope", "field"])
)

display(planned_field_coverage)

print(f"Planned metadata probes: {len(all_probes):,}")
print(
    "Additional observed scalar fields: "
    f"{len(observed_scalar_keys - all_probes):,}"
)

,scope,field,planned_domains,observed_scalar_key,n_files_with_key,n_files_with_informative_scalar,n_files_with_multiple_informative_values,n_reviewed_files
34,sample,days_to_collection,collection_timing,False,0,0,0,649
35,sample,days_to_sample_procurement,collection_timing,False,0,0,0,649
44,sample,time_between_clamping_and_freezing,collection_timing,False,0,0,0,649
45,sample,time_between_excision_and_freezing,collection_timing,False,0,0,0,649
3,analysis,analysis_id,file_processing_dates_not_collection,False,0,0,0,649
9,analysis,workflow_type,file_processing_dates_not_collection,True,649,649,0,649
10,analysis,workflow_version,file_processing_dates_not_collection,True,649,649,0,649
17,file,created_datetime,file_processing_dates_not_collection,False,0,0,0,649
28,file,updated_datetime,file_processing_dates_not_collection,False,0,0,0,649
41,sample,sample_type,generic_sample_descriptors,True,649,649,0,649


Planned metadata probes: 39
Additional observed scalar fields: 11


In [30]:
# =============================================================================
# Assess metadata availability for each RNA-seq source file
# =============================================================================

rna_c_file_rows = []

for row in rna_adjudication_files.itertuples(index=False):
    nodes = rna_c_node_maps[row.file_id]
    record = {
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "rna_multiplicity_class": row.rna_multiplicity_class,
    }

    for domain, probes in RNA_C_FIELDS.items():
        statuses = [
            source_value_status(nodes[scope], key)
            for scope, key in probes
        ]

        record[f"informative_{domain}"] = any(
            status[1] for status in statuses
        )
        record[f"discordant_{domain}"] = any(
            status[2] for status in statuses
        )
        record[f"n_informative_fields_{domain}"] = sum(
            status[1] for status in statuses
        )

    rna_c_file_rows.append(record)

rna_c_file_availability = pd.DataFrame(rna_c_file_rows)

In [31]:
# =============================================================================
# Summarize source metadata availability by multiplicity class
# =============================================================================

rna_c_file_availability = (
    rna_c_file_availability.merge(
        rna_c_exact_path_index[
            ["file_id", "exact_hierarchy_path_rows"]
        ],
        on="file_id",
        validate="one_to_one",
    )
    .sort_values(
        ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"]
    )
    .reset_index(drop=True)
)

summary_fields = [
    f"informative_{domain}" for domain in RNA_C_FIELDS
]

availability_summary = rna_c_file_availability.groupby(
    "rna_multiplicity_class"
)[summary_fields].sum()

display(availability_summary)

,informative_generic_sample_descriptors,informative_collection_timing,informative_sampling_location_detail,informative_processing_provenance,informative_identifiers_not_qc,informative_file_processing_dates_not_collection,informative_quantitative_qc_metadata_probes
rna_multiplicity_class,,,,,,,
MULTIPLE_SAMPLES_IN_CASE,619,0,0,0,619,619,0
ONE_SAMPLE_MULTIPLE_ALIQUOTS,30,0,0,0,30,30,0


In [32]:
# =============================================================================
# Aggregate source metadata availability at case level
# =============================================================================

availability_aggregations = {
    f"n_files_informative_{domain}": (f"informative_{domain}", "sum")
    for domain in RNA_C_FIELDS
}

availability_aggregations.update(
    {
        f"n_files_discordant_{domain}": (f"discordant_{domain}", "sum")
        for domain in RNA_C_FIELDS
    }
)

rna_c_case_coverage = (
    rna_c_file_availability.groupby("case_uuid")
    .agg(
        n_files=("file_id", "nunique"),
        n_exact_hierarchy_rows=("exact_hierarchy_path_rows", "sum"),
        **availability_aggregations,
    )
    .reset_index()
)

In [33]:
# =============================================================================
# Reconcile case-level coverage with audited multiplicity
# =============================================================================

rna_c_case_coverage = rna_c_case_coverage.merge(
    rna_review_cases[
        ["case_uuid", "rna_multiplicity_class", "n_files"]
    ].rename(columns={"n_files": "audited_n_files"}),
    on="case_uuid",
    validate="one_to_one",
)

if (
    len(rna_c_case_coverage) != len(rna_review_cases)
    or not rna_c_case_coverage["n_files"]
    .eq(rna_c_case_coverage["audited_n_files"])
    .all()
):
    raise ValueError("Case-level metadata coverage does not reconcile")

rna_c_case_coverage["selection_policy_status"] = (
    "NO_REPRESENTATIVE_SELECTED"
)

rna_c_case_coverage = rna_c_case_coverage.sort_values(
    "case_uuid"
).reset_index(drop=True)

In [34]:
# =============================================================================
# Summarize case-level metadata availability
# =============================================================================

timing = rna_c_case_coverage["n_files_informative_collection_timing"]

rna_c_case_coverage["has_any_collection_timing"] = timing.gt(0)
rna_c_case_coverage["has_all_collection_timing"] = timing.eq(
    rna_c_case_coverage["n_files"]
)

rna_c_case_coverage["has_any_location_detail"] = (
    rna_c_case_coverage[
        "n_files_informative_sampling_location_detail"
    ].gt(0)
)

rna_c_case_coverage["has_any_quantitative_qc_metadata"] = (
    rna_c_case_coverage[
        "n_files_informative_quantitative_qc_metadata_probes"
    ].gt(0)
)

case_coverage_summary = rna_c_case_coverage.groupby(
    "rna_multiplicity_class"
).agg(
    cases=("case_uuid", "nunique"),
    with_any_collection_timing=("has_any_collection_timing", "sum"),
    with_all_collection_timing=("has_all_collection_timing", "sum"),
    with_any_location_detail=("has_any_location_detail", "sum"),
    with_any_quantitative_qc_metadata=(
        "has_any_quantitative_qc_metadata", "sum"
    ),
)

display(case_coverage_summary)

,cases,with_any_collection_timing,with_all_collection_timing,with_any_location_detail,with_any_quantitative_qc_metadata
rna_multiplicity_class,,,,,
MULTIPLE_SAMPLES_IN_CASE,276,0,0,0,0
ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,0,0,0,0


In [35]:
# =============================================================================
# Verify source record conservation and absence of sample selection
# =============================================================================

if (
    len(rna_c_file_availability) != len(rna_adjudication_files)
    or rna_c_file_availability["file_id"].duplicated().any()
    or rna_c_case_coverage["case_uuid"].duplicated().any()
):
    raise ValueError("Source metadata review altered case or file identities")

if (
    rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_c_case_coverage["selection_policy_status"]
    .ne("NO_REPRESENTATIVE_SELECTED")
    .any()
):
    raise ValueError("Unexpected representative sample selection")

print(f"Cases retained in multiplicity review: {len(rna_c_case_coverage):,}")
print(f"Source files retained:               {len(rna_c_file_availability):,}")
print(
    "Representative files selected:       "
    f"{int(rna_adjudication_files['candidate_selected'].sum()):,}"
)

Cases retained in multiplicity review: 288
Source files retained:               649
Representative files selected:       0


## Targeted GDC Provenance and Specimen Metadata Assessment

Examine a deterministic subset of **12 cases and 29 primary-tumor RNA-seq files** from the 288 cases with unresolved multiplicity.

Use the GDC `/files` and `/cases` endpoints to assess source provenance, case–sample–aliquot relationships, and specimen metadata availability. Preserve the retrieved source responses for reproducibility.

This assessment does not select representative samples, establish relative sequencing quality, or certify donor independence.

In [36]:
# =============================================================================
# Select deterministic cases for targeted GDC provenance assessment
# =============================================================================

RNA_D_PILOT_STRATA = (
    ("MULTIPLE_SAMPLES_IN_CASE", "Uterus, NOS"),
    ("MULTIPLE_SAMPLES_IN_CASE", "Kidney"),
    ("MULTIPLE_SAMPLES_IN_CASE", "Larynx"),
    ("ONE_SAMPLE_MULTIPLE_ALIQUOTS", "Pancreas"),
)

rna_d_pilot_cases = pd.concat(
    [
        rna_adjudication_cases.loc[
            rna_adjudication_cases["rna_multiplicity_class"].eq(kind)
            & rna_adjudication_cases["primary_site_values"].eq(site)
        ]
        .sort_values("case_uuid", kind="stable")
        .head(3)
        for kind, site in RNA_D_PILOT_STRATA
    ],
    ignore_index=True,
)

if len(rna_d_pilot_cases) != 12:
    raise ValueError("Unexpected number of selected provenance cases")

require_unique(rna_d_pilot_cases, ["case_uuid"], "GDC provenance subset")

display(
    rna_d_pilot_cases.groupby(
        ["rna_multiplicity_class", "primary_site_values"]
    )
    .size()
    .rename("n_cases")
    .reset_index()
)

,rna_multiplicity_class,primary_site_values,n_cases
0,MULTIPLE_SAMPLES_IN_CASE,Kidney,3
1,MULTIPLE_SAMPLES_IN_CASE,Larynx,3
2,MULTIPLE_SAMPLES_IN_CASE,"Uterus, NOS",3
3,ONE_SAMPLE_MULTIPLE_ALIQUOTS,Pancreas,3


In [37]:
# =============================================================================
# Identify primary RNA-seq files for the selected provenance cases
# =============================================================================

rna_d_pilot_files = rna_adjudication_files.loc[
    rna_adjudication_files["case_uuid"].isin(
        rna_d_pilot_cases["case_uuid"]
    )
].copy()

rna_d_file_ids = sorted(rna_d_pilot_files["file_id"].tolist())

require_unique(
    rna_d_pilot_files,
    ["file_id"],
    "GDC provenance subset files",
)

print(f"Selected provenance cases: {len(rna_d_pilot_cases):,}")
print(f"Associated primary RNA-seq files: {len(rna_d_file_ids):,}")

Selected provenance cases: 12
Associated primary RNA-seq files: 29


In [38]:
# =============================================================================
# Identify the reusable GDC provenance source snapshot
# =============================================================================

GDC_PILOT_SNAPSHOT_PATH = (
    ROOT
    / "docs"
    / "evidence"
    / "phase1"
    / "notebook113"
    / "113_gdc_multiplicity_pilot_v2_snapshot.json"
)

GDC_FILES_ENDPOINT = "https://api.gdc.cancer.gov/files"
GDC_CASES_ENDPOINT = "https://api.gdc.cancer.gov/cases"

print(f"Source snapshot exists: {GDC_PILOT_SNAPSHOT_PATH.is_file()}")

if GDC_PILOT_SNAPSHOT_PATH.is_file():
    print(f"Source snapshot SHA-256: {sha256(GDC_PILOT_SNAPSHOT_PATH)}")

Source snapshot exists: True
Source snapshot SHA-256: 67c3f4ee86e7799a8ff85c09cb864a42226b9f215930fc092d131767508c7651


In [39]:
# =============================================================================
# Define exact GDC file and case metadata queries
# =============================================================================

gdc_pilot_file_ids = sorted(rna_d_file_ids)
gdc_pilot_case_ids = sorted(rna_d_pilot_cases["case_uuid"].tolist())

GDC_FILE_FIELDS = [
    "file_id", "file_name", "file_size", "md5sum", "access",
    "data_type", "data_category", "cases.case_id",
    "cases.samples.sample_id", "cases.samples.portions.portion_id",
    "cases.samples.portions.analytes.analyte_id",
    "cases.samples.portions.analytes.aliquots.aliquot_id",
    "proportion_reads_mapped", "proportion_reads_duplicated",
    "total_reads", "average_base_quality",
]

GDC_FILE_EXPAND = [
    "associated_entities", "cases.samples",
    "cases.samples.portions", "cases.samples.portions.analytes",
    "cases.samples.portions.analytes.aliquots",
    "analysis", "analysis.metadata.read_groups",
    "analysis.metadata.read_groups.read_group_qcs",
]

In [40]:
# =============================================================================
# Define case-level specimen metadata fields
# =============================================================================

GDC_CASE_FIELDS = [
    "case_id", "samples.sample_id",
    "samples.biospecimen_anatomic_site",
    "samples.biospecimen_laterality",
    "samples.days_to_collection",
    "samples.method_of_sample_procurement",
    "samples.preservation_method",
    "samples.portions.portion_id",
    "samples.portions.analytes.analyte_id",
    "samples.portions.analytes.rna_integrity_number",
    "samples.portions.analytes.aliquots.aliquot_id",
]

GDC_CASE_EXPAND = [
    "samples",
    "samples.portions",
    "samples.portions.analytes",
    "samples.portions.analytes.aliquots",
]

In [41]:
# =============================================================================
# Verify GDC source snapshot query identity
# =============================================================================

gdc_file_query = {
    "filters": {
        "op": "in",
        "content": {
            "field": "file_id",
            "value": gdc_pilot_file_ids,
        },
    },
    "fields": ",".join(GDC_FILE_FIELDS),
    "expand": ",".join(GDC_FILE_EXPAND),
    "format": "JSON",
    "size": len(gdc_pilot_file_ids),
}

gdc_case_query = {
    "filters": {
        "op": "in",
        "content": {
            "field": "case_id",
            "value": gdc_pilot_case_ids,
        },
    },
    "fields": ",".join(GDC_CASE_FIELDS),
    "expand": ",".join(GDC_CASE_EXPAND),
    "format": "JSON",
    "size": len(gdc_pilot_case_ids),
}

gdc_query_contract = {
    "files": gdc_file_query,
    "cases": gdc_case_query,
}

In [42]:
# =============================================================================
# Verify GDC source snapshot provenance and query fingerprint
# =============================================================================

gdc_query_sha = hashlib.sha256(
    json.dumps(
        gdc_query_contract,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

with require_file(GDC_PILOT_SNAPSHOT_PATH).open(
    encoding="utf-8"
) as handle:
    gdc_pilot_snapshot = json.load(handle)

if gdc_pilot_snapshot.get("query_sha256") != gdc_query_sha:
    raise ValueError("GDC snapshot query fingerprint mismatch")

if (
    gdc_pilot_snapshot.get("originating_111_rnaseq_metadata_sha256")
    != expected_sha
):
    raise ValueError("GDC snapshot source fingerprint mismatch")

print("GDC query fingerprint verified:", True)
print("Notebook 111 source fingerprint verified:", True)

GDC query fingerprint verified: True
Notebook 111 source fingerprint verified: True


In [43]:
# =============================================================================
# Verify exact file and case identities in the GDC source snapshot
# =============================================================================

rna_d_hits = gdc_pilot_snapshot.get("hits")
rna_d_case_hits = gdc_pilot_snapshot.get("case_hits")

if not isinstance(rna_d_hits, list) or not isinstance(rna_d_case_hits, list):
    raise ValueError("GDC snapshot has missing or invalid response arrays")

rna_d_hit_by_id = {
    hit.get("file_id") or hit.get("id"): hit
    for hit in rna_d_hits
}

rna_d_case_by_id = {
    hit.get("case_id") or hit.get("id"): hit
    for hit in rna_d_case_hits
}

if (
    len(rna_d_hits) != len(rna_d_hit_by_id)
    or set(rna_d_hit_by_id) != set(gdc_pilot_file_ids)
    or len(rna_d_case_hits) != len(rna_d_case_by_id)
    or set(rna_d_case_by_id) != set(gdc_pilot_case_ids)
):
    raise ValueError("GDC snapshot contains duplicate or unexpected identities")

print(f"Verified GDC file identities: {len(rna_d_hit_by_id):,}")
print(f"Verified GDC case identities: {len(rna_d_case_by_id):,}")

Verified GDC file identities: 29
Verified GDC case identities: 12


In [44]:
# =============================================================================
# Verify source file identities and direct aliquot associations
# =============================================================================

rna_d_identity_by_file = {}

for row in rna_d_pilot_files.itertuples(index=False):
    hit = rna_d_hit_by_id[row.file_id]

    identity_matches = (
        hit.get("file_name") == row.filename
        and str(hit.get("file_size")) == str(row.size_bytes)
        and str(hit.get("md5sum", "")).lower()
        == str(row.provider_md5).lower()
    )

    if not identity_matches:
        raise ValueError(f"GDC source file identity mismatch: {row.file_id}")

    direct = hit.get("associated_entities")
    direct_links = (
        [entity for entity in direct if entity.get("entity_type") == "aliquot"]
        if isinstance(direct, list)
        else []
    )
    exact_links = [
        entity
        for entity in direct_links
        if entity.get("entity_id") == row.aliquot_uuid
        and entity.get("case_id") == row.case_uuid
    ]

    direct_status = (
        "UNAVAILABLE_IN_SOURCE_RESPONSE"
        if not isinstance(direct, list)
        else "EXACT_MATCH"
        if len(direct_links) == len(exact_links) == 1
        else "DIRECT_ASSOCIATION_DISAGREEMENT"
    )

    rna_d_identity_by_file[row.file_id] = {
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "rna_multiplicity_class": row.rna_multiplicity_class,
        "primary_site": row.primary_site,
        "source_file_identity": "EXACT_MATCH",
        "direct_aliquot_association": direct_status,
    }

In [45]:
# =============================================================================
# Resolve exact biospecimen paths across GDC source responses
# =============================================================================

rna_d_nodes = {}

for row in rna_d_pilot_files.itertuples(index=False):
    record = rna_d_identity_by_file[row.file_id]
    hit = rna_d_hit_by_id[row.file_id]
    case_hit = rna_d_case_by_id[row.case_uuid]

    file_paths = exact_source_nodes(
        hit, row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )
    case_paths = exact_source_nodes(
        {"cases": [case_hit]},
        row.case_uuid,
        row.sample_uuid,
        row.aliquot_uuid,
    )

    if record["direct_aliquot_association"] != "EXACT_MATCH":
        chosen_paths, path_source = [], "BLOCKED_DIRECT_FILE_ALIQUOT_LINK"
    elif case_paths:
        chosen_paths, path_source = case_paths, "CASES_ENDPOINT_EXACT_TRIPLET"
    elif file_paths:
        chosen_paths, path_source = file_paths, "FILES_ENDPOINT_EXACT_TRIPLET"
    else:
        chosen_paths, path_source = [], "BLOCKED_NO_EXACT_TRIPLET"

    rna_d_nodes[row.file_id] = {
        scope: [path[scope] for path in chosen_paths]
        for scope in ("sample", "portion", "analyte", "aliquot")
    }

    record.update(
        {
            "n_exact_paths_in_files_response": len(file_paths),
            "n_exact_paths_in_cases_response": len(case_paths),
            "n_exact_hierarchy_paths": len(chosen_paths),
            "exact_biospecimen_path_source": path_source,
        }
    )

In [46]:
# =============================================================================
# Retrieve source analysis metadata and available read-group QC records
# =============================================================================

for file_id, record in rna_d_identity_by_file.items():
    hit = rna_d_hit_by_id[file_id]
    analysis = hit.get("analysis")

    metadata = (
        analysis.get("metadata")
        if isinstance(analysis, dict)
        else None
    )
    read_groups = (
        metadata.get("read_groups") or []
        if isinstance(metadata, dict)
        else []
    )

    if not isinstance(read_groups, list):
        read_groups = []

    qcs = [
        qc
        for group in read_groups
        if isinstance(group, dict)
        for qc in (group.get("read_group_qcs") or [])
        if isinstance(qc, dict)
    ]

    rna_d_nodes[file_id].update(
        {
            "file": [hit],
            "analysis": [analysis] if isinstance(analysis, dict) else [],
            "read_group_qc": qcs,
        }
    )

    record["n_read_groups"] = len(read_groups)
    record["n_read_group_qcs"] = len(qcs)

In [47]:
# =============================================================================
# Summarize verified file-to-biospecimen source relationships
# =============================================================================

rna_d_identity = (
    pd.DataFrame(rna_d_identity_by_file.values())
    .sort_values(["case_uuid", "file_id"])
    .reset_index(drop=True)
)

rna_d_exact_link_n = int(
    rna_d_identity["n_exact_hierarchy_paths"].gt(0).sum()
)

rna_d_exact_link_status = (
    "COMPLETE_EXACT_TRIPLET_RECOVERED"
    if rna_d_exact_link_n == len(rna_d_file_ids)
    else "BLOCKED_INCOMPLETE_EXACT_TRIPLET"
)

source_link_summary = (
    rna_d_identity.assign(
        has_file_hierarchy=lambda df: df[
            "n_exact_paths_in_files_response"
        ].gt(0),
        has_case_hierarchy=lambda df: df[
            "n_exact_paths_in_cases_response"
        ].gt(0),
        has_qc_records=lambda df: df["n_read_group_qcs"].gt(0),
    )
    .groupby(
        ["primary_site", "exact_biospecimen_path_source"]
    )
    .agg(
        n_files=("file_id", "nunique"),
        n_files_with_file_hierarchy=("has_file_hierarchy", "sum"),
        n_files_with_case_hierarchy=("has_case_hierarchy", "sum"),
        n_files_with_qc_records=("has_qc_records", "sum"),
    )
    .reset_index()
)

display(source_link_summary)

print(f"Files with exact biospecimen paths: {rna_d_exact_link_n:,}")
print(f"Files examined:                  {len(rna_d_file_ids):,}")
print(f"Source linkage status:           {rna_d_exact_link_status}")

,primary_site,exact_biospecimen_path_source,n_files,n_files_with_file_hierarchy,n_files_with_case_hierarchy,n_files_with_qc_records
0,Kidney,CASES_ENDPOINT_EXACT_TRIPLET,9,9,9,0
1,Larynx,CASES_ENDPOINT_EXACT_TRIPLET,6,6,6,0
2,Pancreas,CASES_ENDPOINT_EXACT_TRIPLET,8,8,8,0
3,"Uterus, NOS",CASES_ENDPOINT_EXACT_TRIPLET,6,6,6,0


Files with exact biospecimen paths: 29
Files examined:                  29
Source linkage status:           COMPLETE_EXACT_TRIPLET_RECOVERED


In [48]:
# =============================================================================
# Define specimen provenance and molecular preparation probes
# =============================================================================

RNA_D_PROBES = {
    "spatial_specimen": [
        ("sample", "biospecimen_anatomic_site"),
        ("sample", "biospecimen_laterality"),
        ("sample", "tissue_collection_type"),
        ("sample", "specimen_type"),
    ],
    "collection_provenance": [
        ("sample", "days_to_collection"),
        ("sample", "days_to_sample_procurement"),
        ("sample", "method_of_sample_procurement"),
        ("sample", "preservation_method"),
        ("sample", "freezing_method"),
        ("sample", "time_between_excision_and_freezing"),
    ],
    "molecular_preparation": [
        ("portion", "is_ffpe"),
        ("portion", "portion_number"),
        ("analyte", "analyte_type"),
        ("analyte", "rna_integrity_number"),
        ("analyte", "ribosomal_rna_28s_18s_ratio"),
        ("analyte", "a260_a280_ratio"),
        ("analyte", "concentration"),
        ("aliquot", "source_center"),
    ],
}

In [49]:
# =============================================================================
# Define sequencing and read-group QC availability probes
# =============================================================================

RNA_D_PROBES.update(
    {
        "file_level_qc_probes": [
            ("file", "proportion_reads_mapped"),
            ("file", "proportion_reads_duplicated"),
            ("file", "total_reads"),
            ("file", "average_base_quality"),
        ],
        "read_group_qc_probes": [
            ("read_group_qc", "total_sequences"),
            ("read_group_qc", "percent_gc_content"),
            ("read_group_qc", "basic_statistics"),
        ],
    }
)

print(f"Metadata domains: {len(RNA_D_PROBES)}")
print(f"Metadata probes: {sum(map(len, RNA_D_PROBES.values()))}")

Metadata domains: 5
Metadata probes: 25


In [50]:
# =============================================================================
# Assess exact-linked GDC metadata availability
# =============================================================================

rna_d_identity_index = rna_d_identity.set_index("file_id")
rna_d_field_rows = []

for file_id, node_map in rna_d_nodes.items():
    identity = rna_d_identity_index.loc[file_id]

    linked = (
        identity["n_exact_hierarchy_paths"] > 0
        and identity["direct_aliquot_association"] == "EXACT_MATCH"
    )

    for domain, probes in RNA_D_PROBES.items():
        for scope, field in probes:
            nodes = (
                node_map[scope]
                if linked or scope in ("file", "read_group_qc")
                else []
            )

            present, informative, discordant = source_value_status(
                nodes, field
            )

            structured = any(
                isinstance(node.get(field), (dict, list))
                for node in nodes
                if isinstance(node, dict)
            )

            values = sorted(
                {
                    str(node[field]).strip()
                    for node in nodes
                    if isinstance(node, dict)
                    and field in node
                    and informative_scalar(node[field])
                }
            )

            rna_d_field_rows.append(
                {
                    "file_id": file_id,
                    "case_uuid": identity["case_uuid"],
                    "sample_uuid": identity["sample_uuid"],
                    "aliquot_uuid": identity["aliquot_uuid"],
                    "primary_site": identity["primary_site"],
                    "domain": domain,
                    "scope": scope,
                    "field": field,
                    "biospecimen_link_eligible_for_evidence": linked,
                    "key_present": present,
                    "informative_scalar": informative,
                    "informative_values_json": json.dumps(
                        values, ensure_ascii=False
                    ),
                    "structured_value_present": structured,
                    "multiple_informative_values_within_file": discordant,
                    "interpretation": (
                        "AVAILABILITY_ONLY_NOT_A_QC_THRESHOLD"
                        if scope in ("file", "read_group_qc")
                        else "EXACT_LINKED_BIOSPECIMEN_METADATA_ONLY"
                    ),
                }
            )

In [51]:
# =============================================================================
# Summarize GDC specimen and QC metadata availability
# =============================================================================

rna_d_field_evidence = (
    pd.DataFrame(rna_d_field_rows)
    .sort_values(["case_uuid", "file_id", "domain", "field"], kind="stable")
    .reset_index(drop=True)
)

field_availability_summary = (
    rna_d_field_evidence.groupby(["domain", "field"])
    .agg(
        n_files_with_informative_scalar=("informative_scalar", "sum"),
        n_files_with_structured_field=("structured_value_present", "sum"),
        n_files_with_key=("key_present", "sum"),
    )
    .reset_index()
)

display(field_availability_summary)

print(
    "Files with verified biospecimen linkage:",
    rna_d_field_evidence.loc[
        rna_d_field_evidence["biospecimen_link_eligible_for_evidence"],
        "file_id",
    ].nunique(),
)

,domain,field,n_files_with_informative_scalar,n_files_with_structured_field,n_files_with_key
0,collection_provenance,days_to_collection,0,0,12
1,collection_provenance,days_to_sample_procurement,29,0,29
2,collection_provenance,freezing_method,29,0,29
3,collection_provenance,method_of_sample_procurement,29,0,29
4,collection_provenance,preservation_method,29,0,29
5,collection_provenance,time_between_excision_and_freezing,15,0,21
6,file_level_qc_probes,average_base_quality,0,0,0
7,file_level_qc_probes,proportion_reads_duplicated,0,0,0
8,file_level_qc_probes,proportion_reads_mapped,0,0,0
9,file_level_qc_probes,total_reads,0,0,0


Files with verified biospecimen linkage: 29


In [52]:
# =============================================================================
# Compare informative source metadata across files within each case
# =============================================================================

def count_distinct_json_values(values):
    return len({
        item
        for raw in values
        for item in json.loads(raw)
    })


rna_d_case_field_comparison = (
    rna_d_field_evidence.groupby(["case_uuid", "domain", "field"])
    .agg(
        n_files_with_key=("key_present", "sum"),
        n_files_informative=("informative_scalar", "sum"),
        n_files_structured=("structured_value_present", "sum"),
        n_files_without_exact_link=(
            "biospecimen_link_eligible_for_evidence",
            lambda values: int((~values).sum()),
        ),
        n_distinct_informative_values=(
            "informative_values_json",
            count_distinct_json_values,
        ),
    )
    .reset_index()
)

rna_d_case_field_comparison["candidate_values_differ"] = (
    rna_d_case_field_comparison["n_distinct_informative_values"].gt(1)
)

In [53]:
# =============================================================================
# Summarize within-case metadata differences by domain
# =============================================================================

case_field_summary = (
    rna_d_case_field_comparison.assign(
        has_informative_value=lambda df: df["n_files_informative"].gt(0)
    )
    .groupby("domain")
    .agg(
        n_case_fields_informative=("has_informative_value", "sum"),
        n_case_fields_with_distinct_values=(
            "candidate_values_differ",
            "sum",
        ),
    )
    .reset_index()
)

display(case_field_summary)

,domain,n_case_fields_informative,n_case_fields_with_distinct_values
0,collection_provenance,57,0
1,file_level_qc_probes,0,0
2,molecular_preparation,0,0
3,read_group_qc_probes,0,0
4,spatial_specimen,29,0


In [54]:
# =============================================================================
# Reconcile source identities and biospecimen linkage at case level
# =============================================================================

case_linkage = (
    rna_d_identity.assign(
        exact_file_identity=lambda df: df["source_file_identity"].eq(
            "EXACT_MATCH"
        ),
        exact_biospecimen_link=lambda df: df[
            "n_exact_hierarchy_paths"
        ].gt(0),
        has_read_group_qc=lambda df: df["n_read_group_qcs"].gt(0),
    )
    .groupby("case_uuid")
    .agg(
        n_files_with_matching_file_identity=("exact_file_identity", "sum"),
        n_files_with_exact_biospecimen_link=("exact_biospecimen_link", "sum"),
        n_files_with_nested_path=("exact_biospecimen_link", "sum"),
        n_files_with_read_group_qc=("has_read_group_qc", "sum"),
    )
    .reset_index()
)

rna_d_case_summary = rna_d_pilot_cases[
    [
        "case_uuid",
        "primary_site_values",
        "rna_multiplicity_class",
        "n_files",
        "n_samples",
        "n_aliquots",
    ]
].merge(case_linkage, on="case_uuid", validate="one_to_one")

In [55]:
# =============================================================================
# Summarize metadata coverage across pilot cases and tumor contexts
# =============================================================================

informative_files = (
    rna_d_field_evidence.loc[
        rna_d_field_evidence["informative_scalar"]
    ]
    .groupby(["case_uuid", "domain"])["file_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=list(RNA_D_PROBES), fill_value=0)
)

informative_files = informative_files.add_prefix("files_informative_")

rna_d_case_summary = rna_d_case_summary.join(
    informative_files,
    on="case_uuid",
)

coverage_columns = informative_files.columns.tolist()
rna_d_case_summary[coverage_columns] = (
    rna_d_case_summary[coverage_columns].fillna(0).astype("int64")
)

rna_d_case_summary["selection_status"] = "NO_REPRESENTATIVE_SELECTED"

display(
    rna_d_case_summary.groupby(
        ["rna_multiplicity_class", "primary_site_values"]
    )
    .agg(
        n_cases=("case_uuid", "nunique"),
        n_files=("n_files", "sum"),
        exact_nested_path_files=("n_files_with_nested_path", "sum"),
        read_group_qc_files=("n_files_with_read_group_qc", "sum"),
        files_with_spatial_specimen=("files_informative_spatial_specimen", "sum"),
        files_with_collection_provenance=(
            "files_informative_collection_provenance",
            "sum",
        ),
        files_with_molecular_preparation=(
            "files_informative_molecular_preparation",
            "sum",
        ),
        files_with_file_qc_probes=(
            "files_informative_file_level_qc_probes",
            "sum",
        ),
    )
    .reset_index()
)

,rna_multiplicity_class,primary_site_values,n_cases,n_files,exact_nested_path_files,read_group_qc_files,files_with_spatial_specimen,files_with_collection_provenance,files_with_molecular_preparation,files_with_file_qc_probes
0,MULTIPLE_SAMPLES_IN_CASE,Kidney,3,9,9,0,9,9,0,0
1,MULTIPLE_SAMPLES_IN_CASE,Larynx,3,6,6,0,6,6,0,0
2,MULTIPLE_SAMPLES_IN_CASE,"Uterus, NOS",3,6,6,0,6,6,0,0
3,ONE_SAMPLE_MULTIPLE_ALIQUOTS,Pancreas,3,8,8,0,8,8,0,0


In [56]:
# =============================================================================
# Verify provenance assessment scope and absence of sample selection
# =============================================================================

n_cases = len(rna_d_case_summary)
n_files = len(rna_d_identity)
n_probes = sum(len(fields) for fields in RNA_D_PROBES.values())
n_field_records = len(rna_d_field_evidence)
n_case_field_records = len(rna_d_case_field_comparison)

if (
    n_cases != 12
    or n_files != 29
    or rna_d_exact_link_n != n_files
    or n_field_records != n_files * n_probes
    or n_case_field_records != n_cases * n_probes
):
    raise ValueError("GDC provenance assessment scope mismatch")

if (
    rna_d_case_summary["selection_status"].ne(
        "NO_REPRESENTATIVE_SELECTED"
    ).any()
    or rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
):
    raise ValueError("Unexpected representative file selection")

print(f"Cases assessed:                  {n_cases:,}")
print(f"Files with exact linkage:        {rna_d_exact_link_n:,}")
print(f"Metadata observations:           {n_field_records:,}")
print(f"Case-field comparisons:          {n_case_field_records:,}")
print(
    "Representative files selected:  "
    f"{int(rna_adjudication_files['candidate_selected'].sum()):,}"
)

Cases assessed:                  12
Files with exact linkage:        29
Metadata observations:           725
Case-field comparisons:          300
Representative files selected:  0


In [57]:
# =============================================================================
# Load the authenticated STAR Counts technical inventory
# =============================================================================

TECH_112 = META / "112_cptac3_rna_simple_primary_technical_inventory.csv"
HANDOFF_112 = META / "112_cptac3_qc_provenance_handoff.json"

with require_file(HANDOFF_112).open(encoding="utf-8") as handle:
    handoff_112 = json.load(handle)

technical_receipt = (handoff_112.get("outputs") or {}).get(
    "technical_inventory"
)

if not isinstance(technical_receipt, dict):
    raise ValueError("Missing technical inventory provenance in Notebook 112")

if (
    technical_receipt.get("path")
    != TECH_112.relative_to(ROOT).as_posix()
    or technical_receipt.get("rows") != len(ledger)
    or technical_receipt.get("sha256") != sha256(TECH_112)
):
    raise ValueError("Notebook 112 technical inventory fingerprint mismatch")

tech_112 = load_csv(TECH_112)

print(f"Authenticated technical inventory records: {len(tech_112):,}")
print(f"Primary RNA-seq candidate cases:           {len(ledger):,}")

Authenticated technical inventory records: 1,082
Primary RNA-seq candidate cases:           1,082


In [58]:
# =============================================================================
# Verify technical inventory identities against source candidates
# =============================================================================

identity_columns = [
    "case_uuid",
    "file_id",
    "sample_uuid",
    "aliquot_uuid",
]

technical_columns = [
    "n_gene_rows",
    "unstranded_nonnegative_integer_vector",
    "n_invalid_unstranded_gene_values",
    "unstranded_gene_count_total",
    "n_detected_star_genes",
]

missing = set(identity_columns + technical_columns) - set(tech_112.columns)
if missing:
    raise ValueError(f"Missing technical inventory columns: {sorted(missing)}")

require_unique(tech_112, ["case_uuid"], "Technical inventory cases")
require_unique(tech_112, ["file_id"], "Technical inventory files")

matched_identities = tech_112[identity_columns].merge(
    ledger[identity_columns],
    on=identity_columns,
    how="inner",
    validate="one_to_one",
)

if (
    len(tech_112) != len(ledger)
    or len(matched_identities) != len(ledger)
):
    raise ValueError("Technical inventory identities differ from source candidates")

print(f"Source cases with matching technical identities: {len(matched_identities):,}")
print(f"Source cases without matching identities:       {len(ledger) - len(matched_identities):,}")

Source cases with matching technical identities: 1,082
Source cases without matching identities:       0


In [59]:
# =============================================================================
# Incorporate upstream STAR Counts diagnostics into the candidate ledger
# =============================================================================

tech_fields = tech_112[
    ["case_uuid", "file_id", *technical_columns]
].rename(
    columns={
        "n_gene_rows": "source_112_n_gene_rows",
        "unstranded_nonnegative_integer_vector": "source_112_valid_count_vector",
        "n_invalid_unstranded_gene_values": "source_112_invalid_count_values",
        "unstranded_gene_count_total": "source_112_unstranded_count_total",
        "n_detected_star_genes": "source_112_detected_gene_rows",
    }
)

if "source_112_valid_count_vector" in ledger.columns:
    raise ValueError("Technical diagnostics already incorporated into ledger")

ledger = ledger.merge(
    tech_fields,
    on=["case_uuid", "file_id"],
    validate="one_to_one",
)

valid_vectors = ledger["source_112_valid_count_vector"].str.casefold().eq(
    "true"
)
invalid_values = pd.to_numeric(
    ledger["source_112_invalid_count_values"],
    errors="raise",
)

if not valid_vectors.all() or invalid_values.ne(0).any():
    raise ValueError("Upstream STAR Counts diagnostics contain invalid vectors")

duplicate_gene_ids = pd.to_numeric(
    tech_112["n_duplicate_versionless_ids"],
    errors="raise",
)

if duplicate_gene_ids.isna().any() or duplicate_gene_ids.ne(0).any():
    raise ValueError("Duplicate version-normalized STAR gene identifiers")

print(f"Cases with valid unstranded count vectors: {int(valid_vectors.sum()):,}")
print(f"Cases with invalid gene-count values:       {int(invalid_values.gt(0).sum()):,}")

Cases with valid unstranded count vectors: 1,082
Cases with invalid gene-count values:       0


## External Cohort Provenance

Establish source provenance from the audited GDC CPTAC-3 project
and its documented case, sample, aliquot, and file identities.

Classify cohort-level externality relative to the frozen TCGA discovery
population as a **qualified study-design assumption**, supported by
documented source provenance and published study context.

This classification does not establish individual donor independence
or genetically verified non-overlap with TCGA. Donor-level independence
remains explicitly unverified, and any documented contradictory
evidence requires review.

Provenance classification is independent of transcriptomic program
scores and downstream outcomes.

In [60]:
# =============================================================================
# Assign cohort-level external provenance
# =============================================================================

if ledger["project_id"].isna().any():
    raise ValueError("Missing GDC source project identifiers")

ledger["cohort_provenance_status"] = "UNSUPPORTED_SOURCE"
ledger.loc[
    ledger["project_id"].eq("CPTAC-3"),
    "cohort_provenance_status",
] = "COHORT_LEVEL_EXTERNALITY_ASSUMED"

ledger["donor_independence_status"] = "NOT_INDIVIDUALLY_VERIFIED"

provenance_summary = (
    ledger.groupby(
        ["project_id", "cohort_provenance_status", "donor_independence_status"],
        dropna=False,
    )
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
)

display(provenance_summary)

,project_id,cohort_provenance_status,donor_independence_status,n_cases
0,CPTAC-3,COHORT_LEVEL_EXTERNALITY_ASSUMED,NOT_INDIVIDUALLY_VERIFIED,1082


## Cancer Lineage Harmonization

Map CPTAC-3 primary-tumor source contexts to the frozen TCGA cancer lineages using documented anatomical site and disease-type information.

Accept only supported mappings. Ambiguous histological subtypes and unmapped source contexts remain explicitly classified and are not assigned to a TCGA lineage by assumption.

Lineage eligibility is evaluated independently of transcriptomic program scores and downstream outcomes.

In [61]:
# =============================================================================
# Characterize CPTAC-3 source tumor contexts
# =============================================================================

source_keys = ["project_id", "primary_site", "disease_type"]

source_contexts = (
    ledger.groupby(source_keys, dropna=False)
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["primary_site", "disease_type"])
    .reset_index(drop=True)
)

tcga_projects = set(tcga["project_id"].dropna().unique())

print(f"CPTAC-3 source contexts: {len(source_contexts):,}")
print(f"Candidate cases:         {source_contexts['n_cases'].sum():,}")
print(f"Frozen TCGA projects:    {len(tcga_projects):,}")

display(source_contexts)

CPTAC-3 source contexts: 18
Candidate cases:         1,082
Frozen TCGA projects:    33


,project_id,primary_site,disease_type,n_cases
0,CPTAC-3,Brain,Gliomas,159
1,CPTAC-3,Bronchus and lung,Adenomas and Adenocarcinomas,178
2,CPTAC-3,Bronchus and lung,Squamous Cell Neoplasms,89
3,CPTAC-3,Floor of mouth,Squamous Cell Neoplasms,6
4,CPTAC-3,Kidney,Adenomas and Adenocarcinomas,192
5,CPTAC-3,Kidney,Complex Mixed and Stromal Neoplasms,1
6,CPTAC-3,Kidney,Lipomatous Neoplasms,2
7,CPTAC-3,Kidney,Transitional Cell Papillomas and Carcinomas,1
8,CPTAC-3,Larynx,Squamous Cell Neoplasms,18
9,CPTAC-3,Lip,Squamous Cell Neoplasms,2


In [62]:
# =============================================================================
# Define supported CPTAC-3 to TCGA cancer lineage mappings
# =============================================================================

accepted_lineage_rules = [
    ("Bronchus and lung", "Adenomas and Adenocarcinomas", "TCGA-LUAD"),
    ("Bronchus and lung", "Squamous Cell Neoplasms", "TCGA-LUSC"),
    ("Pancreas", "Ductal and Lobular Neoplasms", "TCGA-PAAD"),
    ("Skin", "Nevi and Melanomas", "TCGA-SKCM"),
    ("Stomach", "Adenomas and Adenocarcinomas", "TCGA-STAD"),
    ("Uterus, NOS", "Adenomas and Adenocarcinomas", "TCGA-UCEC"),
    ("Floor of mouth", "Squamous Cell Neoplasms", "TCGA-HNSC"),
    ("Larynx", "Squamous Cell Neoplasms", "TCGA-HNSC"),
    ("Lip", "Squamous Cell Neoplasms", "TCGA-HNSC"),
    (
        "Other and ill-defined sites in lip, oral cavity and pharynx",
        "Squamous Cell Neoplasms",
        "TCGA-HNSC",
    ),
    ("Other and unspecified parts of mouth", "Squamous Cell Neoplasms", "TCGA-HNSC"),
    ("Other and unspecified parts of tongue", "Squamous Cell Neoplasms", "TCGA-HNSC"),
    ("Tonsil", "Squamous Cell Neoplasms", "TCGA-HNSC"),
]

lineage_rules = pd.DataFrame(
    accepted_lineage_rules,
    columns=["primary_site", "disease_type", "tcga_lineage"],
)

require_unique(
    lineage_rules,
    ["primary_site", "disease_type"],
    "Accepted lineage mapping rules",
)

if not set(lineage_rules["tcga_lineage"]).issubset(tcga_projects):
    raise ValueError("Lineage mapping references unavailable TCGA projects")

print(f"Accepted source-context mappings: {len(lineage_rules):,}")
print(f"Supported TCGA lineages:          {lineage_rules['tcga_lineage'].nunique():,}")

Accepted source-context mappings: 13
Supported TCGA lineages:          7


In [63]:
# =============================================================================
# Classify source tumor contexts by lineage mapping support
# =============================================================================

lineage_crosswalk = source_contexts[source_keys].merge(
    lineage_rules,
    on=["primary_site", "disease_type"],
    how="left",
    validate="one_to_one",
)

ambiguous_contexts = (
    (
        lineage_crosswalk["primary_site"].eq("Brain")
        & lineage_crosswalk["disease_type"].eq("Gliomas")
    )
    | (
        lineage_crosswalk["primary_site"].eq("Kidney")
        & lineage_crosswalk["disease_type"].eq(
            "Adenomas and Adenocarcinomas"
        )
    )
)

lineage_crosswalk["lineage_mapping_status"] = "UNMAPPED"
lineage_crosswalk.loc[
    ambiguous_contexts, "lineage_mapping_status"
] = "AMBIGUOUS"
lineage_crosswalk.loc[
    lineage_crosswalk["tcga_lineage"].notna(),
    "lineage_mapping_status",
] = "ACCEPTED"

display(
    lineage_crosswalk.groupby("lineage_mapping_status")
    .size()
    .rename("n_source_contexts")
    .reset_index()
)

,lineage_mapping_status,n_source_contexts
0,ACCEPTED,13
1,AMBIGUOUS,2
2,UNMAPPED,3


In [64]:
# =============================================================================
# Assign supported cancer lineage mappings to candidate cases
# =============================================================================

lineage_fields = [
    *source_keys,
    "tcga_lineage",
    "lineage_mapping_status",
]

if "lineage_mapping_status" in ledger.columns:
    raise ValueError("Lineage mapping already assigned to candidate ledger")

ledger = ledger.merge(
    lineage_crosswalk[lineage_fields],
    on=source_keys,
    how="left",
    validate="many_to_one",
)

if ledger["lineage_mapping_status"].isna().any():
    raise ValueError("Unclassified source tumor contexts")

lineage_case_summary = (
    ledger.groupby(
        ["lineage_mapping_status", "tcga_lineage"],
        dropna=False,
    )
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["lineage_mapping_status", "tcga_lineage"])
)

display(lineage_case_summary)

,lineage_mapping_status,tcga_lineage,n_cases
0,ACCEPTED,TCGA-HNSC,36
1,ACCEPTED,TCGA-LUAD,178
2,ACCEPTED,TCGA-LUSC,89
3,ACCEPTED,TCGA-PAAD,88
4,ACCEPTED,TCGA-SKCM,138
5,ACCEPTED,TCGA-STAD,142
6,ACCEPTED,TCGA-UCEC,56
7,AMBIGUOUS,NaN,351
8,UNMAPPED,NaN,4


## RNA-seq Technical Eligibility and Feature Compatibility

Assess the technical suitability of CPTAC-3 RNA-seq inputs using authenticated source diagnostics and the frozen transcriptomic feature universe.

Distinguish file integrity, raw-count validity, gene coverage, and sequencing QC. Technical compatibility does not imply independent sequencing quality certification or cross-platform expression comparability.

Only cases with supported lineage assignments and compatible source inputs may enter the proposed external cohort.

In [65]:
# =============================================================================
# Load the frozen shared transcriptomic feature universe
# =============================================================================

shared_gene_reference = registry["artifacts"][
    "phase4.400.cross_system_shared_gene_universe"
]

SHARED_GENE_PATH = require_file(ROOT / shared_gene_reference["path"])

if sha256(SHARED_GENE_PATH) != shared_gene_reference["sha256"]:
    raise ValueError("Frozen shared gene universe fingerprint mismatch")

shared_genes = load_csv(SHARED_GENE_PATH)

if "tumor_gene_id" not in shared_genes.columns:
    raise ValueError("Missing frozen tumor gene identifiers")

shared_gene_ids = (
    shared_genes["tumor_gene_id"]
    .str.replace(r"\.\d+$", "", regex=True)
    .str.strip()
)

if len(shared_gene_ids) != 2389 or not shared_gene_ids.is_unique:
    raise ValueError("Unexpected frozen shared gene universe")

print(f"Frozen shared genes: {len(shared_gene_ids):,}")
print(f"Unique gene identifiers: {shared_gene_ids.nunique():,}")
print(f"Reference SHA-256 verified: {True}")

Frozen shared genes: 2,389
Unique gene identifiers: 2,389
Reference SHA-256 verified: True


In [66]:
# =============================================================================
# Resolve STAR Counts paths for lineage-eligible source cases
# =============================================================================

STAR_ROOT = ROOT / "data" / "raw" / "cptac3" / "rnaseq_star_counts"

feature_candidates = ledger.loc[
    ledger["lineage_mapping_status"].eq("ACCEPTED")
].copy()

feature_candidates["star_path"] = feature_candidates.apply(
    lambda row: STAR_ROOT / str(row["file_id"]) / str(row["filename"]),
    axis=1,
)

feature_candidates["star_file_available"] = (
    feature_candidates["star_path"].map(lambda path: path.is_file())
)

print(f"Lineage-eligible files: {len(feature_candidates):,}")
print(f"Files available locally: {int(feature_candidates['star_file_available'].sum()):,}")
print(f"Files missing:           {int((~feature_candidates['star_file_available']).sum()):,}")

Lineage-eligible files: 727
Files available locally: 727
Files missing:           0


In [67]:
# =============================================================================
# Verify frozen feature coverage across lineage-eligible STAR files
# =============================================================================

expected_genes = set(shared_gene_ids)
feature_coverage_records = []

for row in feature_candidates.itertuples(index=False):
    gene_ids = pd.read_csv(
        row.star_path,
        sep="\t",
        comment="#",
        usecols=["gene_id"],
        dtype={"gene_id": "string"},
    )["gene_id"]

    gene_ids = (
        gene_ids.loc[~gene_ids.str.startswith("N_", na=False)]
        .str.replace(r"\.\d+$", "", regex=True)
        .str.strip()
    )

    observed = set(gene_ids.dropna())
    missing = expected_genes - observed

    feature_coverage_records.append(
        {
            "file_id": row.file_id,
            "n_reference_genes": len(expected_genes),
            "n_reference_genes_present": len(expected_genes) - len(missing),
            "n_reference_genes_missing": len(missing),
            "complete_feature_coverage": len(missing) == 0,
        }
    )

feature_coverage = pd.DataFrame(feature_coverage_records)

require_unique(
    feature_coverage,
    ["file_id"],
    "STAR feature compatibility",
)

if len(feature_coverage) != len(feature_candidates):
    raise ValueError("Incomplete STAR feature coverage assessment")

print(f"Files assessed:          {len(feature_coverage):,}")
print(
    "Complete feature coverage: "
    f"{feature_coverage['complete_feature_coverage'].sum():,}"
)
print(
    "Incomplete feature coverage: "
    f"{(~feature_coverage['complete_feature_coverage']).sum():,}"
)
print(
    "Maximum missing genes:   "
    f"{feature_coverage['n_reference_genes_missing'].max():,}"
)

Files assessed:          727
Complete feature coverage: 727
Incomplete feature coverage: 0
Maximum missing genes:   0


In [68]:
# =============================================================================
# Attach frozen feature coverage to the candidate ledger
# =============================================================================

coverage_fields = [
    "file_id",
    "n_reference_genes",
    "n_reference_genes_present",
    "n_reference_genes_missing",
    "complete_feature_coverage",
]

if "complete_feature_coverage" in ledger.columns:
    raise ValueError("Feature coverage already assigned to candidate ledger")

ledger = ledger.merge(
    feature_coverage[coverage_fields],
    on="file_id",
    how="left",
    validate="one_to_one",
)

accepted = ledger["lineage_mapping_status"].eq("ACCEPTED")

if ledger.loc[accepted, "complete_feature_coverage"].isna().any():
    raise ValueError("Missing feature assessment for accepted lineages")

if ledger.loc[~accepted, "complete_feature_coverage"].notna().any():
    raise ValueError("Unexpected feature assessment outside accepted lineages")

print(f"Candidate cases retained: {len(ledger):,}")
print(f"Feature-assessed cases:   {ledger['complete_feature_coverage'].notna().sum():,}")
print(f"Not feature-assessed:     {ledger['complete_feature_coverage'].isna().sum():,}")

Candidate cases retained: 1,082
Feature-assessed cases:   727
Not feature-assessed:     355


In [69]:
# =============================================================================
# Assign external cohort eligibility
# =============================================================================

supported_lineage = ledger["lineage_mapping_status"].eq("ACCEPTED")

supported_source = ledger["cohort_provenance_status"].eq(
    "COHORT_LEVEL_EXTERNALITY_ASSUMED"
)

valid_counts = (
    ledger["source_112_valid_count_vector"]
    .astype("string")
    .str.lower()
    .eq("true")
    & pd.to_numeric(
        ledger["source_112_invalid_count_values"],
        errors="raise",
    ).eq(0)
)

complete_features = ledger["complete_feature_coverage"].eq(True).fillna(False)

eligible = (
    supported_lineage
    & supported_source
    & valid_counts
    & complete_features
)

ledger["external_cohort_eligibility"] = "NOT_ELIGIBLE"
ledger.loc[eligible, "external_cohort_eligibility"] = "ELIGIBLE"

eligibility_summary = (
    ledger.groupby(
        ["external_cohort_eligibility", "lineage_mapping_status"],
        dropna=False,
    )
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
)

display(eligibility_summary)

,external_cohort_eligibility,lineage_mapping_status,n_cases
0,ELIGIBLE,ACCEPTED,727
1,NOT_ELIGIBLE,AMBIGUOUS,351
2,NOT_ELIGIBLE,UNMAPPED,4


In [70]:
# =============================================================================
# Record eligibility reasons and sequencing QC limitations
# =============================================================================

ledger["read_level_qc_status"] = "QC_UNASSESSED"
ledger["eligibility_reason"] = "ELIGIBLE_SOURCE_CANDIDATE"

ledger.loc[
    ledger["lineage_mapping_status"].eq("AMBIGUOUS"),
    "eligibility_reason",
] = "AMBIGUOUS_CANCER_LINEAGE"

ledger.loc[
    ledger["lineage_mapping_status"].eq("UNMAPPED"),
    "eligibility_reason",
] = "UNMAPPED_CANCER_LINEAGE"

unresolved = (
    ledger["external_cohort_eligibility"].eq("NOT_ELIGIBLE")
    & ledger["eligibility_reason"].eq("ELIGIBLE_SOURCE_CANDIDATE")
)

if unresolved.any():
    raise ValueError("Non-eligible cases lack an explicit reason")

## Proposed External Cohort

Construct the external RNA-seq source cohort from cases meeting the predefined
provenance, lineage, and technical eligibility criteria.

Preserve one source file per eligible case, retain source identifiers and
lineage assignments, and verify cohort uniqueness before persistence.

The proposed cohort is intended for lineage-aware external transcriptomic
replication. It does not establish individual donor independence or
cross-platform quantitative comparability.

In [71]:
# =============================================================================
# Construct the eligible external RNA-seq source cohort
# =============================================================================

proposed_cohort = ledger.loc[
    ledger["external_cohort_eligibility"].eq("ELIGIBLE")
].copy()

require_unique(
    proposed_cohort,
    ["case_uuid"],
    "Proposed external cohort cases",
)

require_unique(
    proposed_cohort,
    ["file_id"],
    "Proposed external cohort files",
)

if proposed_cohort["tcga_lineage"].isna().any():
    raise ValueError("Eligible cases contain unresolved cancer lineages")

if not proposed_cohort["complete_feature_coverage"].eq(True).all():
    raise ValueError("Eligible cases have incomplete feature coverage")

cohort_summary = (
    proposed_cohort.groupby("tcga_lineage")
    .agg(
        n_cases=("case_uuid", "nunique"),
        n_files=("file_id", "nunique"),
    )
    .reset_index()
    .sort_values("tcga_lineage")
)

display(cohort_summary)
print(f"Total proposed cases: {len(proposed_cohort):,}")

,tcga_lineage,n_cases,n_files
0,TCGA-HNSC,36,36
1,TCGA-LUAD,178,178
2,TCGA-LUSC,89,89
3,TCGA-PAAD,88,88
4,TCGA-SKCM,138,138
5,TCGA-STAD,142,142
6,TCGA-UCEC,56,56


Total proposed cases: 727


In [72]:
# =============================================================================
# Define the proposed external cohort output schema
# =============================================================================

cohort_export_fields = [
    "project_id",
    "case_uuid",
    "case_submitter_id",
    "sample_uuid",
    "sample_submitter_id",
    "aliquot_uuid",
    "aliquot_submitter_id",
    "file_id",
    "filename",
    "size_bytes",
    "provider_md5",
    "modality",
    "workflow_type",
    "platform",
    "sample_type",
    "tissue_type",
    "primary_site",
    "disease_type",
    "tcga_lineage",
    "cohort_provenance_status",
    "donor_independence_status",
    "other_ambiguous_rna_flag",
]

cohort_export = (
    proposed_cohort[cohort_export_fields]
    .sort_values(["tcga_lineage", "case_uuid"])
    .reset_index(drop=True)
)

require_unique(cohort_export, ["case_uuid"], "External cohort export")
require_unique(cohort_export, ["file_id"], "External cohort source files")

print(f"Cohort cases:   {len(cohort_export):,}")
print(f"Export columns: {len(cohort_export.columns):,}")
print(f"Cancer lineages: {cohort_export['tcga_lineage'].nunique():,}")

display(cohort_export.head(3))

Cohort cases:   727
Export columns: 22
Cancer lineages: 7


,project_id,case_uuid,case_submitter_id,sample_uuid,sample_submitter_id,aliquot_uuid,aliquot_submitter_id,file_id,filename,size_bytes,...,workflow_type,platform,sample_type,tissue_type,primary_site,disease_type,tcga_lineage,cohort_provenance_status,donor_independence_status,other_ambiguous_rna_flag
0,CPTAC-3,0215c1e2-70aa-495b-a1f1-25bd989a9f12,C3N-00204,5bad1cc5-fef1-4c9b-a4e7-193e23923f34,C3N-00204-13,5668bcc3-2d04-48fb-81de-b95154d9cf9f,CPT0313780011,4fdf20d0-d1c3-4b70-9f29-cf6f9beea3ac,ec696c1f-7367-4b8b-888c-4cfaca0b0e28.rna_seq.a...,4261643,...,STAR - Counts,Illumina,Primary Tumor,Tumor,Larynx,Squamous Cell Neoplasms,TCGA-HNSC,COHORT_LEVEL_EXTERNALITY_ASSUMED,NOT_INDIVIDUALLY_VERIFIED,True
1,CPTAC-3,0b5c15c0-6481-4906-b586-8ffb394befd1,C3N-00299,b620794a-4693-40b6-bec9-4312001f9d11,C3N-00299-02,7f775d33-ead4-4cf8-adeb-93c838408f77,CPT0008630009,d82506fc-4bae-4291-a663-b89d86215264,eacd4976-8bb2-4c4c-817d-61cbf926be88.rna_seq.a...,4225332,...,STAR - Counts,Illumina,Primary Tumor,Tumor,Lip,Squamous Cell Neoplasms,TCGA-HNSC,COHORT_LEVEL_EXTERNALITY_ASSUMED,NOT_INDIVIDUALLY_VERIFIED,False
2,CPTAC-3,14a063c2-41c6-4fc0-b334-22b4a0c4c1d5,C3N-00871,933b5bec-0373-451c-b7d5-30bc16ec4b09,C3N-00871-11,d1772abc-fea0-47ec-8093-041d302a9a75,CPT0314480011,3c087a45-36c2-4e42-aec6-0103a369f304,18bb6154-91fe-437a-be9a-108f458d9c83.rna_seq.a...,4252425,...,STAR - Counts,Illumina,Primary Tumor,Tumor,"Other and ill-defined sites in lip, oral cavit...",Squamous Cell Neoplasms,TCGA-HNSC,COHORT_LEVEL_EXTERNALITY_ASSUMED,NOT_INDIVIDUALLY_VERIFIED,True


In [73]:
# =============================================================================
# Define the candidate eligibility ledger output schema
# =============================================================================

ledger_export_fields = [
    "project_id",
    "case_uuid",
    "case_submitter_id",
    "sample_uuid",
    "sample_submitter_id",
    "aliquot_uuid",
    "aliquot_submitter_id",
    "file_id",
    "filename",
    "size_bytes",
    "provider_md5",
    "modality",
    "workflow_type",
    "sample_type",
    "tissue_type",
    "primary_site",
    "disease_type",
    "sample_ancestry_ambiguous",
    "other_ambiguous_rna_flag",
    "primary_candidate_status",
    "cohort_provenance_status",
    "donor_independence_status",
    "tcga_lineage",
    "lineage_mapping_status",
    "source_112_valid_count_vector",
    "source_112_invalid_count_values",
    "complete_feature_coverage",
    "read_level_qc_status",
    "external_cohort_eligibility",
    "eligibility_reason",
]

ledger_export = (
    ledger[ledger_export_fields]
    .sort_values("case_uuid")
    .reset_index(drop=True)
)

require_unique(
    ledger_export,
    ["case_uuid"],
    "Candidate eligibility ledger",
)

require_unique(
    ledger_export,
    ["file_id"],
    "Candidate source files",
)

print(f"Candidate cases: {len(ledger_export):,}")
print(f"Export columns: {len(ledger_export.columns):,}")
print(
    "Eligible cases: "
    f"{ledger_export['external_cohort_eligibility'].eq('ELIGIBLE').sum():,}"
)

Candidate cases: 1,082
Export columns: 30
Eligible cases: 727


In [74]:
# =============================================================================
# Prepare source raw-count compatibility records
# =============================================================================

compatibility_export = proposed_cohort[
    [
        "case_uuid",
        "file_id",
        "source_112_n_gene_rows",
        "source_112_valid_count_vector",
        "source_112_invalid_count_values",
        "n_reference_genes",
        "n_reference_genes_present",
        "n_reference_genes_missing",
        "complete_feature_coverage",
    ]
].copy()

numeric_fields = [
    "source_112_n_gene_rows",
    "source_112_invalid_count_values",
    "n_reference_genes",
    "n_reference_genes_present",
    "n_reference_genes_missing",
]

compatibility_export[numeric_fields] = compatibility_export[
    numeric_fields
].apply(pd.to_numeric, errors="raise")

compatibility_export["raw_count_compatibility_status"] = (
    "COMPATIBLE_RAW_COUNTS"
)

compatibility_export = compatibility_export.sort_values(
    "case_uuid"
).reset_index(drop=True)

require_unique(
    compatibility_export,
    ["case_uuid"],
    "Raw-count compatibility cases",
)

if not compatibility_export["complete_feature_coverage"].eq(True).all():
    raise ValueError("Incomplete frozen-gene coverage")

if not compatibility_export[
    "source_112_valid_count_vector"
].astype("string").str.lower().eq("true").all():
    raise ValueError("Invalid raw-count vectors")

if not compatibility_export[
    "source_112_invalid_count_values"
].eq(0).all():
    raise ValueError("Invalid count values detected")

print(f"Compatible files: {len(compatibility_export):,}")
print(f"Reference genes: {len(shared_gene_ids):,}")
print("Raw-count compatibility: PASS")

Compatible files: 727
Reference genes: 2,389
Raw-count compatibility: PASS


In [75]:
# =============================================================================
# Persist external cohort and source eligibility artifacts
# =============================================================================

identity_fields = ["case_uuid", "file_id", "tcga_lineage"]

eligible_identity = (
    ledger_export.loc[
        ledger_export["external_cohort_eligibility"].eq("ELIGIBLE"),
        identity_fields,
    ]
    .sort_values("case_uuid")
    .reset_index(drop=True)
)

cohort_identity = (
    cohort_export[identity_fields]
    .sort_values("case_uuid")
    .reset_index(drop=True)
)

if not eligible_identity.equals(cohort_identity):
    raise ValueError("Cohort membership differs from the eligibility ledger")

compatible_identity = (
    compatibility_export[["case_uuid", "file_id"]]
    .sort_values("case_uuid")
    .reset_index(drop=True)
)

if not cohort_identity[["case_uuid", "file_id"]].equals(
    compatible_identity
):
    raise ValueError("Raw-count compatibility membership mismatch")

csv_artifacts = {
    "proposed_cohort": (
        cohort_export,
        META / "113_proposed_external_source_cohort.csv",
    ),
    "candidate_eligibility": (
        ledger_export,
        META / "113_primary_candidate_case_eligibility.csv",
    ),
    "raw_count_compatibility": (
        compatibility_export,
        META / "113_raw_count_compatibility.csv",
    ),
}

META.mkdir(parents=True, exist_ok=True)

for name, (frame, path) in csv_artifacts.items():
    frame.to_csv(path, index=False)

    print(
        f"{path.name}: {len(frame):,} rows; "
        f"SHA-256: {sha256(path)}"
    )

113_proposed_external_source_cohort.csv: 727 rows; SHA-256: 75e0a9ee347925fa66adec5f7478d377417d9c943677dce2c7597a8cfc8eb82b
113_primary_candidate_case_eligibility.csv: 1,082 rows; SHA-256: 2d3b24ae6b58b5d4c6c2161fd3dabb40df87973996385ef941621d1fa0763b1c
113_raw_count_compatibility.csv: 727 rows; SHA-256: aac052d3f22803b9489ee90973c3bc98be6ed5e6b2d76b89586fb22e9bd6cc6b


In [76]:
# =============================================================================
# Persist external cohort provenance manifest
# =============================================================================

MANIFEST_PATH = META / "113_source_provenance_manifest.json"

input_paths = {
    "source_file_index": SOURCE_FILE_INDEX,
    "primary_tumor_candidates": PRIMARY_TUMOR_CANDIDATES,
    "acquisition_audit": AUDIT_SUMMARY_PATH,
    "gdc_rnaseq_metadata": RNA_GDC_METADATA_PATH,
    "gdc_provenance_snapshot": GDC_PILOT_SNAPSHOT_PATH,
    "technical_inventory": TECH_112,
    "technical_handoff": HANDOFF_112,
    "tcga_reference": TCGA_REFERENCE_PATH,
    "shared_gene_reference": SHARED_GENE_PATH,
    "artifact_registry": REGISTRY,
}

input_manifest = {
    name: {
        "path": path.relative_to(ROOT).as_posix(),
        "sha256": sha256(path),
    }
    for name, path in input_paths.items()
}

output_manifest = {
    name: {
        "path": path.relative_to(ROOT).as_posix(),
        "sha256": sha256(path),
        "rows": len(frame),
        "columns": len(frame.columns),
    }
    for name, (frame, path) in csv_artifacts.items()
}

lineage_counts = {
    lineage: int(count)
    for lineage, count in cohort_export["tcga_lineage"]
    .value_counts()
    .sort_index()
    .items()
}

manifest = {
    "schema_version": 1,
    "producer": "113_cptac3_external_cohort_construction",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_project": "CPTAC-3",
    "reference_artifacts": {
        "tcga": "phase2.203.final_sample_mapping",
        "shared_genes": "phase4.400.cross_system_shared_gene_universe",
    },
    "source_inputs": input_manifest,
    "derived_outputs": output_manifest,
    "cohort": {
        "candidate_cases": len(ledger_export),
        "proposed_cases": len(cohort_export),
        "lineage_counts": lineage_counts,
        "count_representation": "unstranded",
        "reference_gene_count": len(shared_gene_ids),
        "cohort_provenance_status": "COHORT_LEVEL_EXTERNALITY_ASSUMED",
        "donor_independence_status": "NOT_INDIVIDUALLY_VERIFIED",
    },
    "readiness": {
        "cohort_frozen": False,
        "external_analysis_authorized": False,
        "read_level_qc_status": "QC_UNASSESSED",
        "raw_count_compatibility": "COMPATIBLE_RAW_COUNTS",
        "cross_platform_comparability": "NOT_ESTABLISHED",
    },
    "software": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    handle.write("\n")

print(f"Provenance manifest: {MANIFEST_PATH.name}")
print(f"Proposed source cases: {len(cohort_export):,}")
print(f"Verified output fingerprints: {len(output_manifest)}")
print("Cohort frozen: False")
print("External analysis authorized: False")

Provenance manifest: 113_source_provenance_manifest.json
Proposed source cases: 727
Verified output fingerprints: 3
Cohort frozen: False
External analysis authorized: False


## Results and Scientific Limitations

The proposed external RNA-seq source cohort was constructed from audited
GDC CPTAC-3 primary-tumor records using outcome-blind source eligibility,
cancer-lineage harmonization, and raw-count compatibility criteria.

The final candidate eligibility ledger preserves the complete source
population, including cases without supported lineage assignments and
their non-inclusion reasons.

Eligible source files provide valid STAR `unstranded` raw counts and
complete coverage of the frozen transcriptomic feature universe.

### Interpretation

Cohort-level externality is a qualified study-design assumption.
Individual donor independence from the TCGA discovery population has
not been genetically verified.

Source-file integrity and raw-count compatibility do not establish
independent read-level sequencing QC certification. Cross-platform
expression comparability, tumor-purity effects, proliferation,
biological composition, and potential batch confounding remain
downstream analytical considerations.

### Analytical Readiness

The proposed cohort is technically characterized and traceable through
the persisted source provenance manifest.

It is not a frozen inferential cohort, and external transcriptomic
replication has not been established.

Any subsequent replication analysis requires prospectively defined
normalization, fixed-program projection, lineage-aware inference,
negative controls, and appropriate confounding assessments.